### JAB-Hessian sensitivity estimation & adaptive precision allocation
# GPT-2, C4 calibration, dual-dataset metrics sweep
# Merged: joint QKV + MLP quantization

1. **Calibration on C4** (`allenai/c4`, `en`, streaming `train` split) -- standard GPTQ/AWQ
   protocol, feeding the Hessian, the JAB scoring pass, and the fine-tuning batches alike.
2. **Evaluation on two disjoint corpora**: WikiText-2 test and a held-out C4 `validation`
   slice -- disjoint from calibration by split.
3. **Four metrics per arm per eval dataset**: perplexity, next-token top-1 accuracy, per-layer
   quantization error (`||W-What||_F/||W||_F`, now over BOTH `c_attn` AND `c_fc`/`c_proj`), and
   per-layer reconstruction error (attention-map recon for the QKV block, MLP-output recon for
   the MLP block -- both teacher = float weights, student = quantized weights, same input).
4. **The allocator sees one unit per matrix.** `layer_{i}_Q`, `_K`, `_V`, `_FC`, `_PROJ` each get
   their own Hessian trace (of their own diagonal Hessian block) and their own bit-width, so a
   single layer can give e.g. 4-bit Q, 3-bit K, 8-bit V and 4-bit c_fc / 3-bit c_proj.
5. **One unified sweep** over `MODES = ["uniform", "jab", "joint", "adaptive_joint"]` x
   `SWEEP_BITS = [3, 3.5, 4, 4.5, 6]`. `kl0` is dropped (it was the MSE-only ablation of the
   old attention loss; the QKV loss here is already optionally MSE-only via `LAMBDA_KL=0`).

The scoring loss for the QKV block is the joint attention-aware loss (MSE on merged pre-c_proj
attention output + `lambda_kl` * KL on attention maps) because Q/K/V are genuinely coupled
through the softmax. The scoring loss for the MLP block is MSE-only (there is no attention
distribution to match against once the target is an MLP output).

## Method (sections 0-8)

0. GPT-2 architecture adapter (`AttnConfig` + `MLPConfig`) -- 1. GPTQ core -- 2. Attention-aware
joint loss (QKV) AND MLP MSE loss -- 3. Hutchinson trace estimator -- 4. Greedy
sensitivity-per-cost allocator (one unit per block) -- 5. Model loading & per-layer helpers --
6. Calibration/evaluation data (C4) -- 7. Validation -- 8. Pipeline passes with `collect_metrics`.

## Sweep (sections 9-13)

9. Setup -- 10. fp32 control -- 11. JAB-Hessian scoring (QKV + MLP, once) -- 12. Mode x
bit-budget sweep -- 13. Tidy results table + `results.csv` / `results_per_layer.json`.

In [1]:
!pip install -q transformers datasets

In [ ]:
# --- GPU selection ---
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES set to:", os.environ["CUDA_VISIBLE_DEVICES"])

CUDA_VISIBLE_DEVICES set to: 0


In [3]:
import csv
import gc
import itertools
import json
import math
import os
import random
import time
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as autograd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "gpt2"           # also: "gpt2-medium", "gpt2-large" -- NOT "gpt2-xl", see the
                             # GROUP_SIZE assertion in section 9 (1600 is not a multiple of 128)

FORCE_RERUN = False         # set True to ignore runs/results.json and redo every arm in the
                             # sweep, instead of resuming and skipping arms already recorded

# fp32 compute, fp64 GPTQ -- fp16 activations corrupt every Hessian and the quantized weight
# write-back; fp32 GPTQ compounds rounding into non-monotone perplexity vs. bit-width.
GPU_DTYPE = torch.float32

# --- calibration (C4, not WikiText-2) ---
C4_DATASET_NAME   = "allenai/c4"
C4_DATASET_CONFIG = "en"
CALIB_SEED        = 42
CALIB_N_SAMPLES   = 128
CALIB_SEQ_LEN     = 512
HESSIAN_N_BATCHES = 128     # batches used for H = 2 X^T X -- ALL of them, not a subset.
HUTCH_SAMPLES     = 10      # Hutchinson probes per (layer, block, batch)
JAB_N_BATCHES     = 2       # batches averaged into each block's trace
LAMBDA_KL         = 0.1     # QKV loss = MSE + lambda_kl * KL. Set 0 for MSE-only QKV ablation.
GROUP_SIZE        = 128     # must divide hidden_size (768/1024/1280 for gpt2/-medium/-large)

# --- GPTQ compute choices ---
GPTQ_DTYPE   = torch.float64
PERTURB_MODE = "rtn"        # "rtn" | "gptq" -- allocator's sensitivity-table heuristic only

# --- evaluation: WikiText-2 test AND a disjoint C4 validation slice ---
EVAL_MAX_LENGTH = 1024      # GPT-2's full context (n_positions) -- do not shrink this
EVAL_STRIDE     = 512
C4_EVAL_N_DOCS  = 2000

# --- reconstruction metric eval chunks (both attn-recon and mlp-recon) ---
ATTN_RECON_N_CHUNKS = 8     # eval chunks averaged per (layer, eval dataset), at CALIB_SEQ_LEN

# --- the sweep ---
SWEEP_BITS = [3, 3.5, 4, 4.5, 6]
MODES = ["jab", "jab_raw", "jab_prop", "type_offset", "adaptive_joint"]  # jab_raw = jab WITHOUT group normalization; adaptive_joint = JAB allocation + joint fine-tune (slow, so last). Also available: "uniform", "joint" (uniform + joint FT)
# uniform/joint apply ONE flat bit-width to every (layer, block) unit -- there is no notion of
# a fractional "average" bits for a flat assignment, unlike jab/adaptive_joint's per-unit
# mixing. Fractional SWEEP_BITS entries are left undefined ("--") for those two modes.
# type_offset also mixes bit-widths (so it IS defined at fractional avg_bits), but unlike
# jab/adaptive_joint it runs NO scoring: a per-block-TYPE offset table around a base width,
# then a deterministic per-unit pass to land on the budget (ported from
# run_mistral_quant.py). See section 12.
# jab_prop is jab_assignment_for_normalized_grouped's per-layer knapsack again, but with an
# error-propagation depth weight layered onto the scores first (ported from run_mlp_quant.py's
# PROPAGATION_WEIGHT) -- see section 12.
UNIFORM_AVAILABLE_WIDTHS = {3, 4, 6}

# --- joint fine-tuning ---
JOINT_STEPS_PER_BLOCK = 200
JOINT_LR              = 1e-4   # unused by the FT block itself (see ALPHA_W/ALPHA_S) -- kept as
                                # the default for any other caller of pass_quantize_eval
JOINT_GRAD_CLIP       = 1.0
JOINT_LAYERS          = None   # e.g. range(0, 12, 2); others still get the GPTQ warm start
ALPHA_W               = 0.05   # lr_w = ALPHA_W * scale_flat.mean(); cosine-decayed
ALPHA_S               = 0.02   # lr_s = ALPHA_S * scale_flat.mean() -- LSQ scale lr
SCORE_EVERY           = 10     # score the held-out batch every N FT steps, not every step
LOG_LOSS_COMPONENTS   = False  # print MSE / KL terms separately inside attention_loss (QKV FT)


def free(*objs):
    """
    Run a collection and release cached VRAM. The arguments are only there for readability:
    deleting them inside this function drops *this* frame's references, NOT the caller's
    bindings, so `free(x)` alone never releases `x`.
    """
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def vram(tag=""):
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated() / 1e9
        p = torch.cuda.max_memory_allocated() / 1e9
        print(f"    [vram{' ' + tag if tag else ''}: {a:.2f} GB now, {p:.2f} GB peak this pass]")


def reset_vram_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


print("Device:", DEVICE)
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.1f} GB")
print("Model:", MODEL_ID, "| compute dtype:", GPU_DTYPE, "| GPTQ dtype:", GPTQ_DTYPE)


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition, 102.0 GB
Model: gpt2 | compute dtype: torch.float32 | GPTQ dtype: torch.float64


## 0. GPT-2 architecture adapter

In [4]:
class AttnConfig:
    """
    Shape bookkeeping for GPT-2's fused c_attn (`Q, K, V = c_attn(x).split(hidden, dim=2)`).

    No RoPE (GPT-2 uses learned absolute position embeddings, added once at the embedding
    stage), no GQA (num_kv_heads == num_heads always), no sliding window (plain causal mask).
    """

    def __init__(self, config):
        self.hidden_size   = config.n_embd
        self.num_heads     = config.n_head
        self.num_kv_heads  = config.n_head          # no GQA in GPT-2
        self.head_dim      = config.n_embd // config.n_head
        self.q_out         = self.num_heads * self.head_dim       # == hidden_size
        self.kv_out        = self.num_kv_heads * self.head_dim    # == hidden_size (no GQA shrink)
        self.n_rep         = self.num_heads // self.num_kv_heads  # always 1
        self.scaling       = self.head_dim ** -0.5
        self.n_layers      = config.n_layer

        # Q, K, V are equal thirds of the fused c_attn matrix.
        self.q_numel  = self.hidden_size * self.q_out
        self.kv_numel = self.hidden_size * self.kv_out

    @property
    def qkv_numel(self):
        return self.q_numel + 2 * self.kv_numel

    def __repr__(self):
        return (f"AttnConfig(hidden={self.hidden_size}, layers={self.n_layers}, "
                f"heads={self.num_heads}, head_dim={self.head_dim})")


class MLPConfig:
    """
    Shape bookkeeping for GPT-2's MLP block (`mlp.c_fc`, `mlp.c_proj`).

    c_fc: (hidden, intermediate) "x @ W" Conv1D; c_proj: (intermediate, hidden). Two
    independent matrices with two different, unequal shapes -- unlike Q/K/V, which are three
    equal-sized slices of ONE fused matrix.
    """

    def __init__(self, config):
        self.hidden_size         = config.n_embd
        self.intermediate_size   = config.n_inner if config.n_inner is not None else 4 * config.n_embd
        self.n_layers            = config.n_layer
        self.activation_function = config.activation_function   # informational only

        self.fc_numel   = self.hidden_size * self.intermediate_size
        self.proj_numel = self.intermediate_size * self.hidden_size

    @property
    def mlp_numel(self):
        return self.fc_numel + self.proj_numel

    def __repr__(self):
        return (f"MLPConfig(hidden={self.hidden_size}, intermediate={self.intermediate_size}, "
                f"layers={self.n_layers}, act={self.activation_function})")


def build_causal_mask(T, dtype, device):
    """
    Additive causal mask in the (1, 1, T, T) shape/convention that GPT2Attention's eager
    attention path expects when an `attention_mask` is supplied: 0 on and below the diagonal,
    dtype.min strictly above it.

    Modern `transformers` no longer builds this mask INSIDE `GPT2Attention` from a registered
    causal `self.bias` buffer -- that construction moved up into `GPT2Model.forward`
    (`_update_causal_mask`), which passes the resulting mask down into every block's `attn(...)`
    call. `block_forward` below calls `layer.attn(...)` directly, bypassing `GPT2Model.forward`
    entirely, so without building and passing this mask ourselves, the delegated attention call
    performed NO causal masking at all -- every token could attend to every future token.
    """
    mask = torch.full((T, T), torch.finfo(dtype).min, dtype=dtype, device=device)
    mask = torch.triu(mask, diagonal=1)
    return mask[None, None, :, :]

## 1. GPTQ core

In [5]:
def _quantize_to_grid(w_col, scale, bits):
    """
    Symmetric per-output-row fake quantization of a single input-column (shape: d_out) using a
    fixed per-row scale computed up front from the original weight statistics.
    """
    qmax = 2 ** (bits - 1) - 1
    return torch.clamp(torch.round(w_col / scale), -qmax, qmax) * scale


@torch.no_grad()
def gptq_quantize_layer(weight_in_out, H, bits=4, damp_percent=0.01, group_size=None,
                        act_order=True, return_scale=False, return_compact=False, work_dtype=None):
    """
    Quantizes a weight matrix in the (d_in, d_out) "x @ W" convention using GPTQ.
    Architecture-agnostic: only ever sees a plain (d_in, d_out) matrix and a (d_in, d_in)
    Hessian, so it works equally for c_attn, c_fc, and c_proj.

    weight_in_out: (d_in, d_out) float tensor
    H: (d_in, d_in) Hessian
    damp_percent: Hessian damping for numerical stability (GPTQ default ~0.01)
    group_size: separate per-row scale per contiguous group of `group_size` input columns
    act_order: quantize columns in order of decreasing Hessian diagonal
    return_scale: ALSO return the exact per-(output channel, group) scale used, in the ORIGINAL
        (unpermuted) column order and the same (d_in, d_out) orientation.
    return_compact: additionally return the (d_out, n_groups) compact scale and g_idx.
    work_dtype: fp64 is GPTQ-canonical.

    Returns the fake-quantized weight (also written into weight_in_out in place), or
    (W_final, scale) / (W_final, scale_expanded, scale_compact, g_idx) depending on flags.
    """
    work_dtype = work_dtype or GPTQ_DTYPE
    device = weight_in_out.device
    W = weight_in_out.detach().clone().to(work_dtype).T.contiguous()  # (d_out, d_in)
    d_out, d_in = W.shape

    H = H.clone().to(work_dtype)
    mean_diag = H.diagonal().mean()
    H += damp_percent * mean_diag * torch.eye(d_in, dtype=work_dtype, device=device)

    if act_order:
        perm = torch.argsort(torch.diag(H), descending=True)
        invperm = torch.argsort(perm)
        W = W[:, perm]
        H = H[perm][:, perm]
    else:
        perm = invperm = torch.arange(d_in, device=device)

    H_inv = torch.linalg.inv(H)
    free(H)

    qmax = 2 ** (bits - 1) - 1
    gs = group_size if group_size is not None else d_in
    n_groups = (d_in + gs - 1) // gs
    scale = torch.zeros(d_out, n_groups, dtype=work_dtype, device=device)
    for g in range(n_groups):
        start, end = g * gs, min((g + 1) * gs, d_in)
        scale[:, g] = (W[:, start:end].abs().amax(dim=1) / qmax).clamp(min=1e-8)

    for i in range(d_in):
        row_scale = scale[:, i // gs]
        w_col = W[:, i]
        q_col = _quantize_to_grid(w_col, row_scale, bits)
        err = (w_col - q_col) / H_inv[i, i]
        if i + 1 < d_in:
            W[:, i + 1:] -= torch.outer(err, H_inv[i, i + 1:])
        W[:, i] = q_col
    free(H_inv)

    if act_order:
        W = W[:, invperm]

    W_final = W.T.contiguous().to(weight_in_out.dtype)  # back to (d_in, d_out)
    weight_in_out.copy_(W_final)

    if not return_scale:
        return W_final

    group_idx = torch.arange(d_in, device=device) // gs
    scale_cols = scale[:, group_idx]                                     # (d_out, d_in), permuted
    scale_cols = scale_cols[:, invperm] if act_order else scale_cols
    scale_expanded = scale_cols.T.contiguous().to(weight_in_out.dtype)

    if not return_compact:
        return W_final, scale_expanded

    g_idx = (invperm // gs) if act_order else (torch.arange(d_in, device=device) // gs)
    return W_final, scale_expanded, scale.to(weight_in_out.dtype), g_idx


@torch.no_grad()
def gptq_quantize_conv1d(conv1d, H, bits=4, damp_percent=0.01, group_size=None,
                         act_order=True, return_scale=False, return_compact=False):
    """
    GPTQ for a HF `Conv1D` module (used by GPT-2's c_attn / c_proj / mlp.c_fc), whose weight is
    ALREADY `(d_in, d_out)` -- the forward is `x @ weight + bias`, unlike `nn.Linear`'s
    `(d_out, d_in)` / `x @ weight.T + bias`.

    Guarded with an explicit shape assertion (`gptq_quantize_layer` expects `weight_in_out`'s
    ROW count to equal `H`'s size, i.e. the input dimension).

    Returns W_io (d_in, d_out) [, scale (d_in, d_out)] [, scale_compact (d_out, n_groups), g_idx].
    """
    d_in = H.shape[0]
    assert conv1d.weight.shape[0] == d_in, (
        f"Conv1D weight shape {tuple(conv1d.weight.shape)} does not put the input dimension "
        f"({d_in}) first -- if this fires, the (d_in, d_out) orientation assumption below is "
        f"wrong and gptq_quantize_layer would quantize along the wrong axis")

    W_io = conv1d.weight.data.clone()   # (d_in, d_out), NO transpose
    out = gptq_quantize_layer(W_io, H, bits=bits, damp_percent=damp_percent,
                              group_size=group_size, act_order=act_order,
                              return_scale=return_scale, return_compact=return_compact)
    if return_compact:
        W_q, scale, scale_compact, g_idx = out
        conv1d.weight.data.copy_(W_q.to(conv1d.weight.dtype))
        return W_q, scale, scale_compact, g_idx
    W_q, scale = out if return_scale else (out, None)
    conv1d.weight.data.copy_(W_q.to(conv1d.weight.dtype))
    return (W_q, scale) if return_scale else W_q


def _expand_bits(b, n):
    """Normalize a bit-width spec to a length-`n` tuple of ints. A bare int broadcasts (every
    matrix in the block gets it -- the old per-block behaviour); a sequence is one width per
    matrix, in the block's flat order: (Q, K, V) for the QKV block, (c_fc, c_proj) for MLP."""
    if isinstance(b, (tuple, list)):
        assert len(b) == n, f"expected {n} bit-widths, got {b!r}"
        return tuple(int(x) for x in b)
    return (int(b),) * n


def _fmt_bits(bits, labels):
    """'4 bits' when every matrix shares a width, else e.g. 'Q2/K3/V8 bits'."""
    if len(set(bits)) == 1:
        return f"{bits[0]} bits"
    return "/".join(f"{l}{b}" for l, b in zip(labels, bits)) + " bits"


def _bits_per_weight(bits, sizes, like):
    """Per-WEIGHT bit-width tensor laid out like the block's flat weight vector, so the STE
    fine-tune can put Q, K, V (or c_fc, c_proj) on different grids inside ONE flat tensor."""
    return torch.cat([torch.full((n,), float(b), dtype=like.dtype, device=like.device)
                      for b, n in zip(bits, sizes)])


@torch.no_grad()
def _quantize_qkv_per_slice(c_attn, H, bits3, group_size, damp_percent=0.01,
                            return_scales=False, return_compact=False):
    """
    GPTQ on the Q, K, V output slices of the fused c_attn one at a time, each on its own grid.
    GPTQ never mixes output columns (they only share H, and hence the act_order permutation and
    g_idx), so with equal widths this is identical to the fused call -- checked in section 8's
    validation 7d.
    """
    hs = c_attn.weight.shape[0]
    assert H.shape[0] == hs and c_attn.weight.shape[1] == 3 * hs
    W_all = c_attn.weight.data
    ws, scales, compacts, g_idx = [], [], [], None
    for k, b in enumerate(bits3):
        sl = slice(k * hs, (k + 1) * hs)
        W_k = W_all[:, sl].clone()                                    # (d_in, hs)
        W_q, sc, sc_compact, g = gptq_quantize_layer(
            W_k, H, bits=b, damp_percent=damp_percent, group_size=group_size,
            act_order=True, return_scale=True, return_compact=True)
        W_all[:, sl].copy_(W_q.to(W_all.dtype))
        ws.append(W_q); scales.append(sc); compacts.append(sc_compact)
        g_idx = g if g_idx is None else g_idx          # identical for Q/K/V (same H)
    if not return_scales:
        return W_all.clone()
    if return_compact:
        return ws, scales, torch.cat(compacts, dim=0), g_idx      # compact: (3*hs, n_groups)
    return ws, scales


@torch.no_grad()
def quantize_qkv(c_attn, H, bits, group_size=None, damp_percent=0.01, return_scales=False, return_compact=False):
    """
    GPTQ on GPT-2's fused c_attn, then split into the (W_Q, W_K, W_V) triple the rest of this
    notebook expects.

    `bits` is an int (Q, K, V share it -- ONE fused GPTQ pass, as before) or a
    (bits_Q, bits_K, bits_V) tuple (each slice is quantized on its own grid).
    """
    group_size = GROUP_SIZE if group_size is None else group_size
    bits3 = _expand_bits(bits, 3)

    if len(set(bits3)) > 1:
        return _quantize_qkv_per_slice(c_attn, H, bits3, group_size, damp_percent=damp_percent,
                                       return_scales=return_scales, return_compact=return_compact)

    out = gptq_quantize_conv1d(c_attn, H, bits=bits3[0], damp_percent=damp_percent,
                               group_size=group_size, act_order=True, return_scale=return_scales,
                               return_compact=return_compact)
    if not return_scales:
        return out

    if return_compact:
        W_full, scale_full, scale_compact, g_idx = out
    else:
        W_full, scale_full = out
    hs = W_full.shape[0]                        # d_in == hidden_size; d_out == 3*hidden_size
    ws     = [W_full[:, :hs],     W_full[:, hs:2 * hs],     W_full[:, 2 * hs:]]
    scales = [scale_full[:, :hs], scale_full[:, hs:2 * hs], scale_full[:, 2 * hs:]]
    if return_compact:
        return ws, scales, scale_compact, g_idx   # scale_compact (3*hs, n_groups) stacked Q/K/V
    return ws, scales


@torch.no_grad()
def grid_perturbation(W_io, bits, group_size=None, H=None, mode=None):
    """
    HAWQ-V2's `||Q(W) - W||_F^2`, measured in weight space on GPTQ's own per-(output channel,
    group) symmetric grid. Architecture-agnostic.

    mode="rtn"  : round-to-nearest on that grid. Milliseconds.
    mode="gptq" : the full GPTQ solve (requires H).
    """
    mode = PERTURB_MODE if mode is None else mode
    if mode == "gptq":
        if H is None:
            raise ValueError('mode="gptq" needs the Hessian H')
        W_q = gptq_quantize_layer(W_io.clone(), H, bits=bits, group_size=group_size, act_order=True)
        out = (W_q - W_io).pow(2).sum().item()
        free(W_q)
        return out

    W = W_io.T                                   # (d_out, d_in)
    d_out, d_in = W.shape
    gs = group_size if group_size is not None else d_in
    qmax = 2 ** (bits - 1) - 1
    total = 0.0
    for start in range(0, d_in, gs):
        blk = W[:, start:min(start + gs, d_in)].to(torch.float32)
        scale = (blk.abs().amax(dim=1, keepdim=True) / qmax).clamp(min=1e-8)
        blk_q = torch.clamp(torch.round(blk / scale), -qmax, qmax) * scale
        total += (blk_q - blk).pow(2).sum().item()
    return total


## 2. Losses: attention-aware joint loss (QKV) AND MLP MSE loss

In [6]:
# ---------------------------------------------------------------------------
# QKV weights: flat <-> (W_Q, W_K, W_V), attention forward, joint MSE+KL loss
# ---------------------------------------------------------------------------

def reshape_weights(w_flat, cfg):
    """Flat vector -> Q, K, V in the (d_in, d_out) "x @ W" convention."""
    h, q_n, kv_n = cfg.hidden_size, cfg.q_numel, cfg.kv_numel
    return (w_flat[0:q_n].reshape(h, cfg.q_out),
            w_flat[q_n:q_n + kv_n].reshape(h, cfg.kv_out),
            w_flat[q_n + kv_n:q_n + 2 * kv_n].reshape(h, cfg.kv_out))


def flatten_weights(W_Q, W_K, W_V):
    """Inverse of reshape_weights (all three in (d_in, d_out) orientation)."""
    return torch.cat([W_Q.reshape(-1), W_K.reshape(-1), W_V.reshape(-1)])


def compute_attention(W_Q, W_K, W_V, X, cfg, b_Q=None, b_K=None, b_V=None, attn_mask=None):
    """
    Attention output and attention weights, matching HF's GPT2Attention (eager path) exactly.
    No RoPE, no GQA expansion -- plain causal mask.

    Returns A_hat (batch, seq_len, hidden) -- heads merged, pre-c_proj -- and attn_weights
    (batch, num_heads, seq_len, seq_len), post-softmax.
    """
    B, T, _ = X.shape

    Q = X @ W_Q + (b_Q if b_Q is not None else 0)
    K = X @ W_K + (b_K if b_K is not None else 0)
    V = X @ W_V + (b_V if b_V is not None else 0)

    Q = Q.view(B, T, cfg.num_heads,    cfg.head_dim).transpose(1, 2)
    K = K.view(B, T, cfg.num_kv_heads, cfg.head_dim).transpose(1, 2)
    V = V.view(B, T, cfg.num_kv_heads, cfg.head_dim).transpose(1, 2)

    scores = (Q @ K.transpose(-2, -1)) * cfg.scaling

    if attn_mask is None:
        # Plain bool lower-triangular mask -- matches the source notebook's build_attn_mask.
        attn_mask = torch.ones(T, T, dtype=torch.bool, device=X.device).tril()
    scores = scores.masked_fill(~attn_mask, float("-inf"))

    attn_weights = torch.softmax(scores, dim=-1, dtype=torch.float32).to(Q.dtype)

    A_hat = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, cfg.q_out)
    return A_hat, attn_weights


def mse_loss(w_flat, X, target_A, cfg, b_Q=None, b_K=None, b_V=None, attn_mask=None):
    """L_mse = ||A(X) - A_hat(X)||^2 on merged pre-c_proj attention output."""
    W_Q, W_K, W_V = reshape_weights(w_flat, cfg)
    A_hat, _ = compute_attention(W_Q, W_K, W_V, X, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V,
                                 attn_mask=attn_mask)
    return F.mse_loss(A_hat, target_A)


def kl_loss(w_flat, X, target_attn, cfg, b_Q=None, b_K=None, b_V=None, attn_mask=None):
    """L_kl = KL(target attention maps || predicted attention maps)."""
    W_Q, W_K, W_V = reshape_weights(w_flat, cfg)
    _, attn_weights = compute_attention(W_Q, W_K, W_V, X, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V,
                                        attn_mask=attn_mask)
    log_q = torch.log(attn_weights + 1e-8)
    return F.kl_div(log_q, target_attn, reduction="batchmean")


def attention_loss(w_flat, X, target_A, cfg, target_attn=None, lambda_kl=0.1,
                   b_Q=None, b_K=None, b_V=None, attn_mask=None, log_components=False):
    """MSE only if target_attn is None or lambda_kl == 0; otherwise MSE + lambda_kl * KL."""
    mse = mse_loss(w_flat, X, target_A, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V, attn_mask=attn_mask)
    loss = mse
    kl = None
    if target_attn is not None and lambda_kl:
        kl = kl_loss(w_flat, X, target_attn, cfg, b_Q=b_Q, b_K=b_K, b_V=b_V, attn_mask=attn_mask)
        loss = loss + lambda_kl * kl
    if log_components:
        kl_str = (f", kl={kl.item():.6f}, lambda_kl*kl={lambda_kl * kl.item():.6f}"
                  if kl is not None else ", kl=n/a")
        print(f"    [attention_loss] mse={mse.item():.6f}{kl_str}")
    return loss


# ---------------------------------------------------------------------------
# MLP weights: flat <-> (W_fc, W_proj), MLP forward, MSE-only loss
# ---------------------------------------------------------------------------

def reshape_mlp_weights(w_flat, cfg):
    """Flat vector -> (W_fc, W_proj) in the (d_in, d_out) "x @ W" convention.
    c_fc and c_proj are two independent, UNEQUAL-sized matrices, so this is a plain two-way
    split at `cfg.fc_numel`, not a thirds-split."""
    fc_n = cfg.fc_numel
    return (w_flat[:fc_n].reshape(cfg.hidden_size, cfg.intermediate_size),
            w_flat[fc_n:].reshape(cfg.intermediate_size, cfg.hidden_size))


def flatten_mlp_weights(W_fc, W_proj):
    """Inverse of reshape_mlp_weights (both in (d_in, d_out) orientation)."""
    return torch.cat([W_fc.reshape(-1), W_proj.reshape(-1)])


def compute_mlp(W_fc, W_proj, X, cfg, b_fc=None, b_proj=None, act_fn=None):
    """
    GPT-2 MLP block output, matching HF's GPT2MLP.forward exactly: h -> c_fc -> activation ->
    c_proj. Purely pointwise (per-token) -- no causal mask, no head split.

    W_fc: (hidden, intermediate); W_proj: (intermediate, hidden).
    X: (batch, seq_len, hidden)
    Returns mlp_out (batch, seq_len, hidden) -- the MLP block's contribution to the residual
    stream, pre-add.
    """
    h = X @ W_fc + (b_fc if b_fc is not None else 0)
    h = act_fn(h)
    return h @ W_proj + (b_proj if b_proj is not None else 0)


def mlp_mse_loss(w_flat, X, target_out, cfg, b_fc=None, b_proj=None, act_fn=None):
    """L_mse = ||MLP(X) - MLP_hat(X)||^2 -- MSE-only reconstruction loss for the MLP block."""
    W_fc, W_proj = reshape_mlp_weights(w_flat, cfg)
    mlp_out = compute_mlp(W_fc, W_proj, X, cfg, b_fc=b_fc, b_proj=b_proj, act_fn=act_fn)
    return F.mse_loss(mlp_out, target_out)


# ---------------------------------------------------------------------------
# Where each matrix lives inside its block's flat weight vector
# ---------------------------------------------------------------------------

def qkv_slices(cfg):
    """(start, stop) of Q, K, V inside the flat [W_Q | W_K | W_V] vector."""
    q, kv = cfg.q_numel, cfg.kv_numel
    return {"Q": (0, q), "K": (q, q + kv), "V": (q + kv, q + 2 * kv)}


def mlp_slices(cfg):
    """(start, stop) of c_fc and c_proj inside the flat [W_fc | W_proj] vector."""
    fc, proj = cfg.fc_numel, cfg.proj_numel
    return {"FC": (0, fc), "PROJ": (fc, fc + proj)}


## 3. Hutchinson trace estimator

In [7]:
def hessian_vector_product(loss_fn, params, vector, retain_graph=True):
    # create_graph=True keeps the graph alive for the second grad -- do not release it
    grad = autograd.grad(loss_fn(params), params, create_graph=True, retain_graph=True)[0]
    hvp = autograd.grad(grad, params, grad_outputs=vector, retain_graph=True)[0]
    del grad
    return hvp


def hutchinson_trace_estimator(loss_fn, params, samples=50):
    # trace(H) ~= (1/n) * sum(v_i^T H v_i)
    if not params.requires_grad:
        params.requires_grad_(True)

    estimated_trace = 0.0
    for _ in range(samples):
        # Rademacher vector (as in HAWQ-V2)
        vec = (torch.randint(0, 2, params.shape, device=params.device) * 2 - 1).to(params.dtype)
        hvp = hessian_vector_product(loss_fn, params, vec)
        estimated_trace += torch.dot(vec.flatten(), hvp.flatten())
        del vec, hvp
    return estimated_trace / samples

## 4. Greedy sensitivity-per-cost allocator

**Key change:** an allocator "unit" is now one **matrix**: `layer_{i}_Q`, `_K`, `_V`, `_FC`,
`_PROJ` (5 per layer). Each has its own Hessian trace, its own sensitivity table and its own
bit-width, so a single layer can give e.g. 4-bit Q, 3-bit K, 8-bit V and 4-bit c_fc / 3-bit c_proj.

`MATRIX_GROUP` records which *loss* a unit's trace comes from (`QKV` = attention loss, `MLP` = MLP
MSE). Only units of the same group are on a comparable scale, so score normalization (section 11)
is done per group, not per matrix.


In [8]:
BIT_WIDTHS = [2, 3, 4, 8, 16]


def cost(bits, param_count=1.0):
    """
    Memory cost of one unit at a given bit-width. `param_count=1.0` keeps `budget` directly
    readable as "average bits per (layer, block) unit". The QKV and MLP blocks in one GPT-2
    layer have different parameter counts (QKV: 3*hidden^2 = 1.77M for gpt2; MLP: 8*hidden^2
    = 4.72M for gpt2), so this is a heuristic -- for a strict parameter-weighted budget,
    pass `param_count=block_numel` instead. Default 1.0 matches the source notebooks.
    """
    return bits * param_count


def greedy_allocate(scores, budget, param_counts=None, min_bits=None):
    """
    scores: {unit_name: {bits: sensitivity}};  budget: max total cost.
    param_counts: optional {unit_name: numel}. Without it every unit is weighted 1.0, so the
      knapsack trades bits off per UNIT, not per PARAMETER -- for this notebook's QKV (1.77M)
      vs. MLP (4.72M) blocks that's a ~2.7x mismatch, and it means "avg_bits" doesn't reflect
      the model's real average bit-width. Pass real block sizes (see `jab_assignment_for`) to
      get a parameter-weighted allocation, which spends bits where they buy the most accuracy
      per byte instead of per unit -- this is what actually improves perplexity at a given
      real memory budget.
    min_bits: optional hard FLOOR -- no unit may be downgraded below this width, regardless
      of what the sensitivity-per-cost ratio says. 2-bit quantization error is a cliff, not a
      slope: a single unit pushed to 2 bits can cost more perplexity than every other unit's
      savings buy back, and the greedy ratio test cannot see that because the sensitivity table
      is smooth in `bits` by construction. Passing e.g. min_bits=3 forbids the cliff while
      leaving the allocator full freedom to vary between 3/4/8/16. This stacks ON TOP of any
      score normalization -- it constrains the search space, it does not rescale the scores.
      If the budget cannot be met without breaking the floor, the floor wins and the allocation
      comes back over budget (with a warning) rather than silently dropping to 2 bits.
      1. Start every unit at the HIGHEST bit-width.
      2. While over budget, apply the downgrade with the best cost-saved-per-accuracy-lost ratio.
      3. Spend leftover budget on the best affordable upgrades.
    """
    param_counts = param_counts or {}
    floor = min(BIT_WIDTHS) if min_bits is None else min_bits

    def pc(name):
        return param_counts.get(name, 1.0)

    current_bits = {name: max(BIT_WIDTHS) for name in scores}

    def total_cost():
        return sum(cost(current_bits[n], pc(n)) for n in scores)

    def total_sensitivity():
        return sum(scores[n][current_bits[n]] for n in scores)

    while total_cost() > budget:                     # downgrade loop
        best = best_bits = best_ratio = None
        for name in scores:
            lower = [b for b in BIT_WIDTHS if b < current_bits[name] and b >= floor]
            if not lower:
                continue
            nb = max(lower)
            saved = cost(current_bits[name], pc(name)) - cost(nb, pc(name))
            added = scores[name][nb] - scores[name][current_bits[name]]
            ratio = added / saved
            if best_ratio is None or ratio < best_ratio:
                best_ratio, best, best_bits = ratio, name, nb
        if best is None:
            break
        current_bits[best] = best_bits

    if total_cost() > budget:
        print(f"  [greedy_allocate] WARNING: budget {budget:.4g} is not reachable with "
              f"min_bits={floor} -- every unit is already at the floor and total cost is "
              f"{total_cost():.4g}. Lower min_bits or raise avg_bits.")

    made_an_upgrade = True                           # spend any remaining budget
    while made_an_upgrade:
        made_an_upgrade = False
        best = best_bits = best_ratio = None
        for name in scores:
            higher = [b for b in BIT_WIDTHS if b > current_bits[name]]
            if not higher:
                continue
            nb = min(higher)
            extra = cost(nb, pc(name)) - cost(current_bits[name], pc(name))
            if total_cost() + extra > budget:
                continue
            ratio = (scores[name][current_bits[name]] - scores[name][nb]) / extra
            if best_ratio is None or ratio > best_ratio:
                best_ratio, best, best_bits = ratio, name, nb
        if best is not None:
            current_bits[best] = best_bits
            made_an_upgrade = True

    return current_bits, total_cost(), total_sensitivity()


def brute_force_optimal(scores, budget, max_units=8):
    """Exhaustive search, for sanity-checking the greedy allocator on a small subset."""
    names = list(scores.keys())
    if len(names) > max_units:
        raise ValueError(f"brute_force_optimal over {len(names)} units = "
                         f"{len(BIT_WIDTHS)}^{len(names)} combinations. Restrict `scores` to at "
                         f"most {max_units} units, or raise max_units if you really mean it.")
    best_assignment = best_cost = None
    best_sensitivity = float("inf")
    for combo in itertools.product(*([BIT_WIDTHS] * len(names))):
        c = sum(cost(b) for b in combo)
        if c > budget:
            continue
        s = sum(scores[names[i]][combo[i]] for i in range(len(names)))
        if s < best_sensitivity:
            best_sensitivity, best_cost = s, c
            best_assignment = dict(zip(names, combo))
    return best_assignment, best_cost, best_sensitivity


def scores_to_allocator_format(jab_scores, bit_widths=None):
    """Heuristic table (trace / bits^1.5), kept as an alternative to the HAWQ-V2 measured form."""
    bit_widths = BIT_WIDTHS if bit_widths is None else bit_widths
    return {n: {b: t / (b ** 1.5) for b in bit_widths} for n, t in jab_scores.items()}


# ---- allocator units: ONE PER MATRIX -------------------------------------------------
MATRICES = ("Q", "K", "V", "FC", "PROJ")        # per layer: W_Q, W_K, W_V, c_fc, c_proj
QKV_MATRICES = ("Q", "K", "V")                  # order == flat [W_Q | W_K | W_V]
MLP_MATRICES = ("FC", "PROJ")                   # order == flat [W_fc | W_proj]
MATRIX_GROUP = {"Q": "QKV", "K": "QKV", "V": "QKV",   # which LOSS the trace comes from ->
                "FC": "MLP", "PROJ": "MLP"}           # which units are directly comparable


def unit_name(i, m):
    """Allocator unit name, e.g. unit_name(3, 'K') -> 'layer_3_K'."""
    return f"layer_{i}_{m}"


def layer_idx_of(name):
    return int(name.split("_")[1])


def matrix_of(name):
    """'Q', 'K', 'V', 'FC' or 'PROJ' from an allocator unit name."""
    return name.rsplit("_", 1)[-1]


def block_of(name):
    """Loss group of a unit: 'QKV' (attention loss) or 'MLP' (MLP MSE loss)."""
    return MATRIX_GROUP[matrix_of(name)]


def bits_fns_from_assignment(assignment):
    """{layer_{i}_{Q,K,V,FC,PROJ}: bits} -> (bits_fn_qkv, bits_fn_mlp) for pass_quantize_eval.
    Each returns a per-matrix TUPLE: (Q, K, V) and (c_fc, c_proj)."""
    bits_fn_qkv = lambda i: tuple(assignment[unit_name(i, m)] for m in QKV_MATRICES)
    bits_fn_mlp = lambda i: tuple(assignment[unit_name(i, m)] for m in MLP_MATRICES)
    return bits_fn_qkv, bits_fn_mlp


## 5. Model loading & per-layer helpers

In [9]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, AutoConfig
from datasets import load_dataset


def load_model_and_tokenizer(model_id, device):
    """
    Loads a full, resident GPT-2 and returns a BRAND NEW model every call. This matters: the
    pipeline quantizes c_attn.weight / mlp.c_fc.weight / mlp.c_proj.weight IN PLACE, so reusing
    one resident model across arms compounds each arm's damage into the next.
    """
    tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained(model_id).to(device).to(GPU_DTYPE)
    model.config._attn_implementation = "eager"
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model, tokenizer


def load_decoder_layer(reader, config, idx):
    """Index into `reader.transformer.h` -- `reader` is the resident model."""
    return reader.transformer.h[idx]


# ---------- QKV accessors ----------

def qkv_of(layer):
    """GPT-2 has one fused QKV module, not three separate projections."""
    return layer.attn.c_attn


def qkv_weights_io(layer, dtype=None):
    """The layer's Q/K/V in the (d_in, d_out) convention, split out of the fused c_attn matrix.
    `c_attn.weight` is already (d_in, d_out) = (hidden, 3*hidden) -- Conv1D, not nn.Linear, so
    there is NO transpose here."""
    W = layer.attn.c_attn.weight
    hs = W.shape[0]
    W_Q, W_K, W_V = W[:, :hs], W[:, hs:2 * hs], W[:, 2 * hs:]
    if dtype:
        return W_Q.to(dtype), W_K.to(dtype), W_V.to(dtype)
    return W_Q, W_K, W_V


def qkv_bias_io(layer, dtype=None):
    """GPT-2's c_attn bias, split the same way as the weight. Biases are NEVER a GPTQ/STE target."""
    b = layer.attn.c_attn.bias
    hs = b.shape[0] // 3
    b_Q, b_K, b_V = b[:hs], b[hs:2 * hs], b[2 * hs:]
    if dtype:
        return b_Q.to(dtype), b_K.to(dtype), b_V.to(dtype)
    return b_Q, b_K, b_V


@torch.no_grad()
def write_qkv_flat(layer, w_flat, cfg):
    """Scatter a flat [W_Q|W_K|W_V] vector back into c_attn's weight, in place."""
    W_Q, W_K, W_V = reshape_weights(w_flat, cfg)
    hs = cfg.hidden_size
    W = layer.attn.c_attn.weight.data
    W[:, :hs].copy_(W_Q.to(W.dtype))
    W[:, hs:2 * hs].copy_(W_K.to(W.dtype))
    W[:, 2 * hs:].copy_(W_V.to(W.dtype))


# ---------- MLP accessors ----------

def mlp_of(layer):
    """GPT-2's MLP block for this layer (`c_fc`, `c_proj`, and the activation function as
    `.act`). GPT-2's MLP is already two separate modules -- no fused matrix to reach through."""
    return layer.mlp


def mlp_weights_io(layer, dtype=None):
    """The layer's (W_fc, W_proj) in the (d_in, d_out) convention. Both `c_fc` and `c_proj` are
    HF Conv1D modules, already (d_in, d_out) -- no transpose."""
    W_fc = layer.mlp.c_fc.weight
    W_proj = layer.mlp.c_proj.weight
    if dtype:
        return W_fc.to(dtype), W_proj.to(dtype)
    return W_fc, W_proj


def mlp_bias_io(layer, dtype=None):
    """`c_fc`/`c_proj`'s biases."""
    b_fc = layer.mlp.c_fc.bias
    b_proj = layer.mlp.c_proj.bias
    if dtype:
        return b_fc.to(dtype), b_proj.to(dtype)
    return b_fc, b_proj


@torch.no_grad()
def write_mlp_flat(layer, w_flat, cfg):
    """Scatter a flat [W_fc|W_proj] vector back into c_fc/c_proj's weights, in place."""
    W_fc, W_proj = reshape_mlp_weights(w_flat, cfg)
    layer.mlp.c_fc.weight.data.copy_(W_fc.to(layer.mlp.c_fc.weight.dtype))
    layer.mlp.c_proj.weight.data.copy_(W_proj.to(layer.mlp.c_proj.weight.dtype))


# ---------- forward / hooks ----------

def block_forward(layer, h, cfg_attn, attn_mask, w_qkv=None, return_attn=False):
    """
    One GPT-2 block, run manually for the QKV/attention piece but delegating everything else --
    both LayerNorms, c_proj, the MLP -- to the real resident modules. Used by section 7's
    validation and nowhere else. The joint fine-tune uses `block_forward_flat` below instead,
    which needs gradients to reach a flat MLP weight vector too and so cannot delegate.
    """
    x = layer.ln_1(h)
    b_Q, b_K, b_V = qkv_bias_io(layer)
    if w_qkv is not None:
        W_Q, W_K, W_V = w_qkv
    else:
        W_Q, W_K, W_V = qkv_weights_io(layer)
    A, attn_w = compute_attention(W_Q, W_K, W_V, x, cfg_attn, b_Q=b_Q, b_K=b_K, b_V=b_V,
                                  attn_mask=attn_mask)
    attn_out = layer.attn.c_proj(A.to(layer.attn.c_proj.weight.dtype))
    h = h + attn_out
    h = h + layer.mlp(layer.ln_2(h))
    return (h, attn_w) if return_attn else (h, None)


def _layernorm32(ln, x):
    """`ln` applied in float32 regardless of the module's own parameter dtype. Used by
    `block_forward_flat` so the joint fine-tune's forward stays in fp32 end to end (GPU_DTYPE is
    fp32 here anyway, but this keeps the helper safe if that ever changes)."""
    return F.layer_norm(x, ln.normalized_shape, ln.weight.to(torch.float32),
                        ln.bias.to(torch.float32), ln.eps)


def block_forward_flat(layer, h, cfg_attn, cfg_mlp, attn_mask, *,
                       w_qkv_flat=None, w_mlp_flat=None,
                       w_qkv_io=None, w_mlp_io=None,
                       b_qkv=None, b_mlp=None, act_fn=None, return_attn=False):
    """
    ONE COMPLETE GPT-2 block -- ln_1 -> attention -> attn.c_proj -> residual add -> ln_2 -> MLP
    -> residual add -- computed in float32 and DIFFERENTIABLE w.r.t. whichever flat weight
    vectors are passed in. This is what the joint fine-tune in `pass_quantize_eval` optimizes
    against: the block's true end-to-end hidden-state output, not either sub-block's isolated
    reconstruction target.

    Differs from `block_forward` above (which is section 7's validation helper and delegates
    c_proj / ln_2 / the MLP to the resident modules, so gradients can't reach a *flat* MLP
    weight vector): here every matmul that a fine-tuned weight participates in is written out
    explicitly, so autograd reaches both `w_qkv_flat` and `w_mlp_flat`.

    Weight source per block, in priority order:
      1. `w_*_flat`   -- a flat vector (the STE-quantized student). Differentiable.
      2. `w_*_io`     -- an explicit (d_in, d_out) tuple, e.g. the float teacher snapshot.
      3. neither      -- the layer's CURRENT live weights. For a block that is quantized but not
                         being tuned this is its post-GPTQ weight, and for a block that is not
                         quantized at all it is the original float weight; either way that block
                         stays FIXED while still participating in the same block forward, so the
                         loss reflects the real end-to-end output.

    attn.c_proj, both LayerNorms and all biases are never quantization targets in this notebook,
    so they are always taken from the live module (as float32 constants).

    Returns (h_out, attn_weights), attn_weights being None unless `return_attn`.
    """
    h = h.to(torch.float32)
    if b_qkv is None:
        b_qkv = qkv_bias_io(layer, dtype=torch.float32)
    if b_mlp is None:
        b_mlp = mlp_bias_io(layer, dtype=torch.float32)
    act_fn = layer.mlp.act if act_fn is None else act_fn

    # ---- attention half ----
    if w_qkv_flat is not None:
        W_Q, W_K, W_V = reshape_weights(w_qkv_flat, cfg_attn)
    elif w_qkv_io is not None:
        W_Q, W_K, W_V = w_qkv_io
    else:
        W_Q, W_K, W_V = qkv_weights_io(layer, dtype=torch.float32)

    x1 = _layernorm32(layer.ln_1, h)
    A, attn_w = compute_attention(W_Q, W_K, W_V, x1, cfg_attn,
                                  b_Q=b_qkv[0], b_K=b_qkv[1], b_V=b_qkv[2], attn_mask=attn_mask)
    W_ap = layer.attn.c_proj.weight.to(torch.float32)
    b_ap = layer.attn.c_proj.bias.to(torch.float32)
    h1 = h + (A @ W_ap + b_ap)

    # ---- MLP half ----
    if w_mlp_flat is not None:
        W_fc, W_proj = reshape_mlp_weights(w_mlp_flat, cfg_mlp)
    elif w_mlp_io is not None:
        W_fc, W_proj = w_mlp_io
    else:
        W_fc, W_proj = mlp_weights_io(layer, dtype=torch.float32)

    x2 = _layernorm32(layer.ln_2, h1)
    h2 = h1 + compute_mlp(W_fc, W_proj, x2, cfg_mlp, b_fc=b_mlp[0], b_proj=b_mlp[1],
                          act_fn=act_fn)
    return (h2, attn_w) if return_attn else (h2, None)


@torch.no_grad()
def embed_cache(reader, ids_list, device=None, dtype=None):
    """Hidden states entering layer 0, for every sequence -- used only by section 7's validation.
    `wte(ids) + wpe(positions)`: GPT-2 bakes ALL position information in once, here."""
    device = device or DEVICE
    dtype = dtype or GPU_DTYPE
    wte = reader.transformer.wte.weight.to(device=device, dtype=dtype)
    wpe = reader.transformer.wpe.weight.to(device=device, dtype=dtype)
    out = []
    for ids in ids_list:
        ids = ids.to(device)
        T = ids.shape[1]
        pos = torch.arange(T, device=device)
        out.append(F.embedding(ids, wte) + F.embedding(pos, wpe)[None])
    free(wte, wpe)
    return out


def collect_hessian_via_hook(model, module, calibration_batches, device):
    """
    Registers a forward PRE-hook on `module`, runs `calibration_batches` through the WHOLE model
    in no_grad mode, and returns the accumulated Hessian H = 2 X^T X for that module.

    It is ALSO what makes the sequential-GPTQ property automatic: `model` is the SAME object
    `pass_quantize_eval` is quantizing layer by layer, so by the time this runs for layer i,
    blocks 0..i-1 already hold their final (possibly quantized) weights.

    `calibration_batches` should be an iterable of input_ids tensors of shape (1, seq_len).
    Returns H, a (d_in, d_in) float64 tensor.
    """
    d_in = module.weight.shape[0]           # Conv1D weight is (in_features, out_features)
    H = torch.zeros(d_in, d_in, dtype=torch.float64, device=device)
    n_samples = [0]

    def _hook(mod, inputs):
        x = inputs[0].detach()
        x = x.reshape(-1, x.shape[-1]).to(torch.float64)   # (tokens, d_in)
        H.add_(2.0 * x.T @ x)
        n_samples[0] += x.shape[0]

    handle = module.register_forward_pre_hook(_hook)
    try:
        model.eval()
        with torch.no_grad():
            for input_ids in calibration_batches:
                model(input_ids.to(device))
    finally:
        handle.remove()

    if n_samples[0] > 0:
        H /= n_samples[0]
    return H


def capture_module_input(model, module, batch, device):
    """
    One-shot capture of the tensor entering `module` from a REAL forward pass of `model` on
    `batch`, via a temporary forward pre-hook.
    """
    captured = {}

    def _hook(mod, inputs):
        captured["X"] = inputs[0].detach().to(torch.float32)

    handle = module.register_forward_pre_hook(_hook)
    try:
        with torch.no_grad():
            model(batch.to(device))
    finally:
        handle.remove()
    return captured["X"]


# ---------- evaluation ----------

@torch.no_grad()
def _sliding_window_nll(model, ids, max_length, stride):
    """Sliding-window NLL sum and scored-token count over a single 1D token-id tensor."""
    device = model.device
    ids = ids.to(device)
    seq_len = ids.shape[0]
    model.eval()
    nll_sum, n_tokens, prev_end = 0.0, 0, 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = ids[begin:end].unsqueeze(0)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        out = model(input_ids, labels=target_ids)
        nll_sum += out.loss.item() * trg_len
        n_tokens += trg_len
        prev_end = end
        if end == seq_len:
            break
    return nll_sum, n_tokens


@torch.no_grad()
def _sliding_window_nll_acc(model, ids, max_length, stride):
    """Same windowing/masking as _sliding_window_nll, plus next-token top-1 accuracy over the
    IDENTICAL positions in the same forward pass."""
    device = model.device
    ids = ids.to(device)
    seq_len = ids.shape[0]
    model.eval()
    nll_sum, correct, n_tokens, prev_end = 0.0, 0, 0, 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = ids[begin:end].unsqueeze(0)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        out = model(input_ids, labels=target_ids)
        nll_sum += out.loss.item() * trg_len
        shift_logits = out.logits[:, :-1, :]
        shift_targets = target_ids[:, 1:]
        mask = shift_targets != -100
        preds = shift_logits.argmax(dim=-1)
        correct += ((preds == shift_targets) & mask).sum().item()
        n_tokens += trg_len
        prev_end = end
        if end == seq_len:
            break
    return nll_sum, correct, n_tokens


@torch.no_grad()
def evaluate_perplexity_and_accuracy(model, tokenizer, text, max_length=None, stride=None):
    """Perplexity + next-token top-1 accuracy over the SAME strided windows, in ONE forward pass.
    `text` is a raw corpus string -- WikiText-2 test or a C4 slice."""
    max_length = EVAL_MAX_LENGTH if max_length is None else max_length
    stride = EVAL_STRIDE if stride is None else stride
    ids = tokenizer(text, return_tensors="pt").input_ids[0]
    nll_sum, correct, n_tokens = _sliding_window_nll_acc(model, ids, max_length, stride)
    return math.exp(nll_sum / n_tokens), correct / n_tokens


## 6. Calibration and evaluation data (C4)

In [10]:
from datasets import load_dataset


def build_calibration_ids(tokenizer, n_samples=None, seq_len=None, seed=None):
    """
    `n_samples` chunks of `seq_len` tokens from C4 (en, train, streaming) -- the standard
    GPTQ/AWQ calibration protocol. A document must tokenize to >= seq_len tokens to contribute
    a chunk; the chunk is a random contiguous span within it.
    """
    n_samples = CALIB_N_SAMPLES if n_samples is None else n_samples
    seq_len = CALIB_SEQ_LEN if seq_len is None else seq_len
    seed = CALIB_SEED if seed is None else seed
    assert seq_len <= 1024, f"seq_len={seq_len} exceeds GPT-2's context length (1024)"

    rng = random.Random(seed)
    stream = load_dataset(C4_DATASET_NAME, C4_DATASET_CONFIG, split="train", streaming=True)
    stream = stream.shuffle(seed=seed, buffer_size=10_000)

    out = []
    for ex in stream:
        if len(out) >= n_samples:
            break
        ids = tokenizer(ex["text"], return_tensors="pt", add_special_tokens=False).input_ids[0]
        if ids.shape[0] < seq_len:
            continue
        start = rng.randint(0, ids.shape[0] - seq_len)
        out.append(ids[start:start + seq_len].unsqueeze(0))
    assert len(out) == n_samples, f"only found {len(out)}/{n_samples} C4 documents long enough"
    return out


def load_wikitext2_eval_text():
    """Full WikiText-2 test split, joined."""
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    return "\n\n".join(t for t in raw["text"] if t.strip())


def load_c4_eval_text(n_docs=None, seed=None):
    """C4 `validation` split, streamed and shuffled with a fixed seed, `n_docs` documents joined
    into one eval corpus. Disjoint from calibration's `train` stream by construction."""
    n_docs = C4_EVAL_N_DOCS if n_docs is None else n_docs
    seed = CALIB_SEED if seed is None else seed
    stream = load_dataset(C4_DATASET_NAME, C4_DATASET_CONFIG, split="validation", streaming=True)
    stream = stream.shuffle(seed=seed, buffer_size=10_000)
    texts = [ex["text"] for ex, _ in zip(stream, range(n_docs))]
    return "\n\n".join(t for t in texts if t.strip())


def build_metric_eval_chunks(tokenizer, text, n_chunks, seq_len, seed):
    """`n_chunks` random contiguous `seq_len`-token spans from `text` -- the fixed-size batches
    the attention-reconstruction and MLP-reconstruction metrics average over."""
    rng = random.Random(seed)
    ids = tokenizer(text, return_tensors="pt").input_ids[0]
    out = []
    for _ in range(n_chunks):
        if ids.shape[0] <= seq_len:
            out.append(ids[:seq_len].unsqueeze(0))
            continue
        start = rng.randint(0, ids.shape[0] - seq_len)
        out.append(ids[start:start + seq_len].unsqueeze(0))
    return out


## 7. Validation

In [11]:
def _validate_gpt2_forward(verbose=True):
    """
    7a: hand-written block_forward (QKV/attention piece) vs. a real GPT2LMHeadModel, on a tiny
        random model.
    7b: _sliding_window_nll (the core of evaluate_perplexity) vs. HF's own labels=... loss.
    Plus: the Conv1D orientation guard, and the STE-grid-is-a-no-op-on-GPTQ's-own-output check.
    """
    from transformers import GPT2Config, GPT2LMHeadModel

    torch.manual_seed(0)
    tiny = GPT2Config(vocab_size=256, n_embd=32, n_head=4, n_layer=3, n_positions=64,
                      bos_token_id=0, eos_token_id=0)
    ref = GPT2LMHeadModel(tiny)
    ref.config._attn_implementation = "eager"
    ref.eval()
    for p in ref.parameters():
        p.requires_grad_(False)

    cfg = AttnConfig(ref.config)
    mlp_cfg = MLPConfig(ref.config)
    assert (cfg.num_heads, cfg.num_kv_heads, cfg.head_dim, cfg.n_rep) == (4, 4, 8, 1)
    assert cfg.q_out == cfg.kv_out == 32, "Q/K/V must be equal-sized thirds (no GQA shrink)"
    assert mlp_cfg.intermediate_size == 4 * mlp_cfg.hidden_size, (
        "tiny config's n_inner should default to 4x hidden -- check MLPConfig against the real "
        "GPT2Block/GPT2MLP fallback")

    T = 24
    ids = torch.randint(0, 256, (1, T))
    with torch.no_grad():
        want = ref(ids).logits

    old_device, old_dtype = globals()["DEVICE"], globals()["GPU_DTYPE"]
    globals()["DEVICE"], globals()["GPU_DTYPE"] = "cpu", torch.float32
    try:
        with torch.no_grad():
            h = embed_cache(ref, [ids])[0]
            attn_mask = torch.ones(T, T, dtype=torch.bool, device=h.device).tril()
            for i in range(cfg.n_layers):
                h, attn_w = block_forward(ref.transformer.h[i], h, cfg, attn_mask, return_attn=True)
                assert attn_w.shape == (1, cfg.num_heads, T, T)
            got = ref.lm_head(ref.transformer.ln_f(h))

        rel = ((got - want).norm() / want.norm()).item()
        if verbose:
            print(f"  7a: hand-written forward vs. GPT2LMHeadModel.forward: relative error "
                  f"{rel:.3e}")
        assert rel < 1e-3, (
            f"hand-written forward disagrees with the real model (rel err {rel:.3e}). Check the "
            f"Conv1D orientation, the c_attn Q/K/V split order, and the causal mask.")

        # 7b: _sliding_window_nll vs. HF's own labels=... loss.
        nll_ours, n_ours = _sliding_window_nll(ref, ids[0], max_length=T, stride=T)
        ppl_ours = math.exp(nll_ours / n_ours)
        with torch.no_grad():
            ppl_hf = math.exp(ref(ids, labels=ids).loss.item())
        rel_ppl = abs(ppl_ours - ppl_hf) / ppl_hf
        if verbose:
            print(f"  7b: _sliding_window_nll vs. HF's labels=... loss: ours={ppl_ours:.4f} "
                  f"hf={ppl_hf:.4f} (rel err {rel_ppl:.3e})")
        assert rel_ppl < 1e-3
    finally:
        globals()["DEVICE"], globals()["GPU_DTYPE"] = old_device, old_dtype

    # STE grid no-op on GPTQ's own output
    W0 = torch.randn(32, 32) * 0.05
    Xd = torch.randn(2000, 32)
    Hd = (2.0 * Xd.T @ Xd / Xd.shape[0]).double()
    Wq, sc = gptq_quantize_layer(W0.clone(), Hd, bits=4, group_size=8, act_order=True,
                                 return_scale=True, work_dtype=torch.float64)
    qmax = 7
    rel_ste = ((torch.clamp(torch.round(Wq / sc), -qmax, qmax) * sc - Wq).norm() / Wq.norm()).item()
    assert rel_ste < 1e-6, f"return_scale is not the grid GPTQ used (rel err {rel_ste:.3e})"
    if verbose:
        print(f"  GPTQ return_scale is exact (re-quantization relative error {rel_ste:.3e})")

    # Conv1D orientation guard
    class _FakeConv1D:
        def __init__(self, w):
            self.weight = torch.nn.Parameter(w.clone())

    d_in, d_out = 16, 48
    fake = _FakeConv1D(torch.randn(d_in, d_out) * 0.05)
    Xc = torch.randn(500, d_in)
    Hc = (2.0 * Xc.T @ Xc / Xc.shape[0]).double()
    W_before_shape = tuple(fake.weight.data.shape)
    W_after = gptq_quantize_conv1d(fake, Hc, bits=4, group_size=8)
    assert tuple(W_after.shape) == W_before_shape == (d_in, d_out)
    _ = Xc @ fake.weight.data     # must not raise a shape error
    if verbose:
        print(f"  gptq_quantize_conv1d preserves the (d_in, d_out) Conv1D orientation "
              f"{tuple(fake.weight.data.shape)}, and x @ W still type-checks")

    free(ref)
    return rel


print("Validation 7a/7b: hand-written forward + perplexity path vs. real GPT2LMHeadModel...")
_validate_gpt2_forward()
print("PASSED -- the hand-written forward and the perplexity path reproduce GPT2LMHeadModel.\n")


def _validate_adaptive_joint_pass(verbose=True):
    """
    7c: pass_quantize_eval(bits_fn=..., finetune=True) on a hand-made mixed 2/3/4/8-bit
    assignment, tiny random GPT-2, CPU-only, seconds not minutes. Exercises exactly the
    section-8 code path (adaptive allocation + joint fine-tuning together).

    The mixed assignment here is per-BLOCK: each of (QKV, MLP) per tiny layer gets its own
    bit-width, so this check exercises both blocks' quantization + FT paths.
    """
    from transformers import GPT2Config, GPT2LMHeadModel

    torch.manual_seed(1)
    tiny = GPT2Config(vocab_size=256, n_embd=32, n_head=4, n_layer=4, n_positions=64,
                      bos_token_id=0, eos_token_id=0)
    ref = GPT2LMHeadModel(tiny)
    ref.config._attn_implementation = "eager"
    ref.eval()
    for p in ref.parameters():
        p.requires_grad_(False)

    cfg = AttnConfig(ref.config)
    mlp_cfg = MLPConfig(ref.config)

    # Per-block mixed assignment: one bit-width per (layer, block). 2-bit is included
    # deliberately -- the allocator can and does emit it.
    mixed_bits_qkv = {0: 2, 1: 3, 2: 4, 3: 8}
    mixed_bits_mlp = {0: 4, 1: 2, 2: 3, 3: 8}
    assert set(mixed_bits_qkv) == set(range(cfg.n_layers))
    assert set(mixed_bits_mlp) == set(range(mlp_cfg.n_layers))

    def bits_fn_qkv(i):
        return mixed_bits_qkv[i]

    def bits_fn_mlp(i):
        return mixed_bits_mlp[i]

    calib_ids = [torch.randint(0, 256, (1, 16)) for _ in range(4)]
    eval_ids = torch.randint(0, 256, (48,))

    def synthetic_eval_fn():
        nll, n = _sliding_window_nll(ref, eval_ids, max_length=16, stride=16)
        return math.exp(nll / n)

    old_device, old_dtype = globals()["DEVICE"], globals()["GPU_DTYPE"]
    globals()["DEVICE"], globals()["GPU_DTYPE"] = "cpu", torch.float32
    try:
        ppl, notes = pass_quantize_eval(
            ref, ref.config, cfg, mlp_cfg, calib_ids, synthetic_eval_fn,
            bits_fn_qkv=bits_fn_qkv, bits_fn_mlp=bits_fn_mlp, finetune=True,
            steps_per_block=2, n_hessian_batches=2, group_size=8,
            label="7c: mixed-bit adaptive+joint (tiny CPU model, QKV+MLP)", verbose=False,
            return_notes=True)
    finally:
        globals()["DEVICE"], globals()["GPU_DTYPE"] = old_device, old_dtype

    assert math.isfinite(ppl) and ppl > 0, f"non-finite/degenerate perplexity: {ppl}"
    # notes is a tuple of (qkv_notes, mlp_notes); each is {layer_idx: (bits, committed)}
    assert isinstance(notes, tuple) and len(notes) == 2, "return_notes must return (qkv, mlp)"
    qkv_notes, mlp_notes = notes
    assert set(qkv_notes) == set(mixed_bits_qkv)
    assert set(mlp_notes) == set(mixed_bits_mlp)
    for i, bits in mixed_bits_qkv.items():
        got_bits, committed = qkv_notes[i]
        assert got_bits == bits, f"QKV layer {i}: note recorded {got_bits} bits, assigned {bits}"
        assert committed in (True, False), (
            f"QKV layer {i} ({bits}-bit): finetune=True but committed={committed!r}")
    for i, bits in mixed_bits_mlp.items():
        got_bits, committed = mlp_notes[i]
        assert got_bits == bits, f"MLP layer {i}: note recorded {got_bits} bits, assigned {bits}"
        assert committed in (True, False), (
            f"MLP layer {i} ({bits}-bit): finetune=True but committed={committed!r}")

    if verbose:
        qkv_detail = ", ".join(f"L{i}={b}b/{'FT' if c else 'GPTQ'}"
                               for i, (b, c) in sorted(qkv_notes.items()))
        mlp_detail = ", ".join(f"L{i}={b}b/{'FT' if c else 'GPTQ'}"
                               for i, (b, c) in sorted(mlp_notes.items()))
        print(f"  mixed-bit adaptive+joint pass ran end-to-end on a tiny CPU GPT-2 model "
              f"(ppl={ppl:.3f})")
        print(f"    QKV: {qkv_detail}")
        print(f"    MLP: {mlp_detail}")

    free(ref)
    return ppl


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Validation 7a/7b: hand-written forward + perplexity path vs. real GPT2LMHeadModel...
  7a: hand-written forward vs. GPT2LMHeadModel.forward: relative error 0.000e+00


  7b: _sliding_window_nll vs. HF's labels=... loss: ours=255.8802 hf=255.8802 (rel err 0.000e+00)
  GPTQ return_scale is exact (re-quantization relative error 2.487e-08)
  gptq_quantize_conv1d preserves the (d_in, d_out) Conv1D orientation (16, 48), and x @ W still type-checks
PASSED -- the hand-written forward and the perplexity path reproduce GPT2LMHeadModel.



## 8. The pipeline passes

Two `bits_fn` callables are passed to `pass_quantize_eval`: `bits_fn_qkv` and `bits_fn_mlp`, one
per block. Each returns either an int (every matrix in the block shares it, the old behaviour) or a
**per-matrix tuple**: `(bits_Q, bits_K, bits_V)` for QKV, `(bits_c_fc, bits_c_proj)` for MLP.
`pass_score` now returns one trace / sensitivity table per matrix.


In [12]:
class STEQuantize(torch.autograd.Function):
    """Straight-through on w; LSQ gradient on scale (Esser et al.), scaled by 1/sqrt(n_weights*qmax)
    since each group's scale now accumulates gradient from n_weights shared weights.
    `bits` may be an int or a per-weight tensor (same shape as w) -- the latter is how Q/K/V
    (or c_fc/c_proj) get different grids inside one flat weight vector."""

    @staticmethod
    def forward(ctx, w, scale, bits, n_weights):
        qmax = 2 ** (bits - 1) - 1
        v = w / scale
        q = torch.clamp(torch.round(v), -qmax, qmax)
        ctx.save_for_backward(v, q)
        ctx.qmax = qmax
        ctx.grad_scale_factor = (n_weights * qmax) ** -0.5
        return q * scale

    @staticmethod
    def backward(ctx, grad_output):
        v, q = ctx.saved_tensors
        qmax = ctx.qmax
        grad_scale_local = torch.where(v.abs() <= qmax, q - v, torch.sign(v) * qmax)
        return grad_output, grad_output * grad_scale_local * ctx.grad_scale_factor, None, None


def ste_quantize(w, scale, bits, n_weights):
    return STEQuantize.apply(w, scale, bits, n_weights)


def expand_scale_qkv(scale_compact, g_idx, cfg):
    """(3*hidden, n_groups) scale, one per (output row, group), shared g_idx (d_in,) -> a
    per-weight tensor flattened in the same [Q|K|V] order as flatten_weights/w_flat."""
    hs = cfg.hidden_size
    parts = [chunk[:, g_idx].T.reshape(-1)
             for chunk in (scale_compact[:hs], scale_compact[hs:2 * hs], scale_compact[2 * hs:])]
    return torch.cat(parts)


def expand_scale_mlp(scale_compact_fc, g_idx_fc, scale_compact_proj, g_idx_proj, cfg):
    """Per-weight scale tensor matching flatten_mlp_weights' [W_fc | W_proj] order."""
    fc_flat = scale_compact_fc[:, g_idx_fc].T.reshape(-1)
    proj_flat = scale_compact_proj[:, g_idx_proj].T.reshape(-1)
    return torch.cat([fc_flat, proj_flat])


def _matrix_trace(loss_full, w_full, start, stop, samples):
    """
    Hutchinson trace of ONE matrix's diagonal Hessian block: the Hessian of `loss_full` w.r.t.
    w_full[start:stop] alone, every other weight in the block frozen at its float value.

    tr(H) = sum of the diagonal blocks' traces, so for the QKV block trace_Q + trace_K + trace_V
    equals the old joint QKV trace (up to Hutchinson noise). Same for c_fc + c_proj.
    """
    head = w_full[:start].detach()
    tail = w_full[stop:].detach()
    p = w_full[start:stop].detach().clone().requires_grad_(True)

    def loss_fn(x):
        return loss_full(torch.cat([head, x, tail]))

    t = hutchinson_trace_estimator(loss_fn, p, samples=samples).item()
    del p
    return t


def pass_score(reader, config, cfg_attn, cfg_mlp, calib_ids, *, samples=None, n_score_batches=None,
               n_hessian_batches=None, lambda_kl=None, group_size=None, bit_widths=None,
               perturb_mode=None, verbose=True):
    """
    Scoring pass over an all-float model. `reader` MUST be a freshly-loaded, unquantized model.

    Scores EACH MATRIX separately: Q, K, V, c_fc, c_proj (5 units per layer). For a matrix the
    trace is that of the Hessian block for its own weights, all other weights held at float:
      - Q / K / V: joint attention-aware loss (MSE on merged pre-c_proj attention output
        + lambda_kl * KL on attention maps), differentiated w.r.t. only that matrix.
      - c_fc / c_proj: MLP MSE reconstruction loss, differentiated w.r.t. only that matrix.

    Returns (jab_scores, allocator_input):
      jab_scores      -- {layer_{i}_{Q,K,V,FC,PROJ}: Hessian trace}
      allocator_input -- {layer_{i}_{Q,K,V,FC,PROJ}: {bits: trace * perturbation}}, where the
                         perturbation is ||Q(W) - W||_F^2 of THAT matrix at that bit-width.
    """
    samples = HUTCH_SAMPLES if samples is None else samples
    n_score_batches = JAB_N_BATCHES if n_score_batches is None else n_score_batches
    n_hessian_batches = HESSIAN_N_BATCHES if n_hessian_batches is None else n_hessian_batches
    lambda_kl = LAMBDA_KL if lambda_kl is None else lambda_kl
    group_size = GROUP_SIZE if group_size is None else group_size
    bit_widths = BIT_WIDTHS if bit_widths is None else bit_widths
    perturb_mode = PERTURB_MODE if perturb_mode is None else perturb_mode

    reset_vram_peak()
    # bool lower-triangular mask (matches the source notebook's build_attn_mask).
    mask = torch.ones(calib_ids[0].shape[1], calib_ids[0].shape[1],
                      dtype=torch.bool, device=DEVICE).tril()
    q_sl, m_sl = qkv_slices(cfg_attn), mlp_slices(cfg_mlp)

    jab_scores, allocator_input = {}, {}
    print(f"Scoring {cfg_attn.n_layers} layers x {len(MATRICES)} matrices (Q,K,V,c_fc,c_proj): "
          f"{samples} Hutchinson probes x {n_score_batches} batches per matrix, "
          f"Q/K/V loss = MSE + {lambda_kl}*KL, c_fc/c_proj loss = MSE, "
          f"perturbation mode '{perturb_mode}'")

    for i in range(cfg_attn.n_layers):
        layer = load_decoder_layer(reader, config, i)
        tables = {}

        # ============================================================
        # Q, K, V: joint attention-aware loss, one trace per matrix
        # ============================================================
        H_qkv = collect_hessian_via_hook(reader, layer.attn.c_attn,
                                         calib_ids[:n_hessian_batches], DEVICE)
        W32_qkv = qkv_weights_io(layer, dtype=torch.float32)       # (W_Q, W_K, W_V)
        b32_qkv = qkv_bias_io(layer, dtype=torch.float32)
        w_qkv = flatten_weights(*W32_qkv)
        traces_qkv = {m: [] for m in QKV_MATRICES}
        for b in calib_ids[:n_score_batches]:
            X = capture_module_input(reader, layer.attn.c_attn, b, DEVICE)
            with torch.no_grad():
                tA, tattn = compute_attention(*W32_qkv, X, cfg_attn,
                                              b_Q=b32_qkv[0], b_K=b32_qkv[1], b_V=b32_qkv[2],
                                              attn_mask=mask)
            tattn = tattn if lambda_kl else None

            def loss_full_qkv(w):
                return attention_loss(w, X, tA, cfg_attn, target_attn=tattn, lambda_kl=lambda_kl,
                                      b_Q=b32_qkv[0], b_K=b32_qkv[1], b_V=b32_qkv[2],
                                      attn_mask=mask)

            for m in QKV_MATRICES:
                s0, e0 = q_sl[m]
                traces_qkv[m].append(_matrix_trace(loss_full_qkv, w_qkv, s0, e0, samples))
            free(X, tA, tattn)

        for k, m in enumerate(QKV_MATRICES):
            name = unit_name(i, m)
            trace = sum(traces_qkv[m]) / len(traces_qkv[m])
            jab_scores[name] = trace
            tables[name] = {bits: trace * grid_perturbation(W32_qkv[k], bits, group_size=group_size,
                                                            H=H_qkv, mode=perturb_mode)
                            for bits in bit_widths}
        allocator_input.update(tables)
        del H_qkv, W32_qkv, w_qkv
        free()

        # ============================================================
        # c_fc, c_proj: MLP MSE loss, one trace per matrix
        # ============================================================
        H_mlp = {"FC": collect_hessian_via_hook(reader, layer.mlp.c_fc,
                                                calib_ids[:n_hessian_batches], DEVICE),
                 "PROJ": collect_hessian_via_hook(reader, layer.mlp.c_proj,
                                                  calib_ids[:n_hessian_batches], DEVICE)}
        W32_mlp = mlp_weights_io(layer, dtype=torch.float32)        # (W_fc, W_proj)
        b32_mlp = mlp_bias_io(layer, dtype=torch.float32)
        act_fn = layer.mlp.act
        w_mlp = flatten_mlp_weights(*W32_mlp)
        traces_mlp = {m: [] for m in MLP_MATRICES}
        for b in calib_ids[:n_score_batches]:
            X = capture_module_input(reader, layer.mlp.c_fc, b, DEVICE)
            with torch.no_grad():
                target_out = compute_mlp(*W32_mlp, X, cfg_mlp, b_fc=b32_mlp[0],
                                        b_proj=b32_mlp[1], act_fn=act_fn)

            def loss_full_mlp(w):
                return mlp_mse_loss(w, X, target_out, cfg_mlp, b_fc=b32_mlp[0],
                                    b_proj=b32_mlp[1], act_fn=act_fn)

            for m in MLP_MATRICES:
                s0, e0 = m_sl[m]
                traces_mlp[m].append(_matrix_trace(loss_full_mlp, w_mlp, s0, e0, samples))
            free(X, target_out)

        for k, m in enumerate(MLP_MATRICES):
            name = unit_name(i, m)
            trace = sum(traces_mlp[m]) / len(traces_mlp[m])
            jab_scores[name] = trace
            tables[name] = allocator_input[name] = {
                bits: trace * grid_perturbation(W32_mlp[k], bits, group_size=group_size,
                                                H=H_mlp[m], mode=perturb_mode)
                for bits in bit_widths}
        del H_mlp, W32_mlp, b32_mlp, w_mlp
        free()

        if verbose:
            for m in MATRICES:
                name = unit_name(i, m)
                tr = jab_scores[name]
                lvl = "HIGH" if tr > 100 else "MEDIUM" if tr > 10 else "LOW"
                print(f"  {name}: trace={tr:.4f} [{lvl}]  " +
                      ", ".join(f"{b}b={v:.3e}" for b, v in tables[name].items()))
            sq = sum(jab_scores[unit_name(i, m)] for m in QKV_MATRICES)
            sm = sum(jab_scores[unit_name(i, m)] for m in MLP_MATRICES)
            print(f"  layer {i}: Q+K+V trace sum = {sq:.4f}, c_fc+c_proj trace sum = {sm:.4f}")

        del layer
        free()

    free(mask)
    vram("after scoring pass")
    return jab_scores, allocator_input


In [13]:
def pass_quantize_eval(reader, config, cfg_attn, cfg_mlp, calib_ids, eval_fn, *,
                       bits_fn_qkv=None, bits_fn_mlp=None,
                       finetune=False, n_hessian_batches=None, group_size=None, lambda_kl=None,
                       steps_per_block=None, lr=None, grad_clip_norm=None, layer_indices=None,
                       label="", verbose=True, return_notes=False,
                       collect_metrics=False, metric_eval_chunks=None):
    """
    Quantize-and-evaluate pass. `reader` MUST be a freshly-loaded model whenever either bits_fn
    is not None -- quantization mutates weights in place.

    bits_fn_qkv: None -> leave c_attn untouched; else layer_idx -> bit-width(s) for the QKV block:
        an int (Q, K, V share it) or a tuple (bits_Q, bits_K, bits_V).
    bits_fn_mlp: None -> leave mlp untouched; else layer_idx -> bit-width(s) for the MLP block:
        an int (c_fc and c_proj share it) or a tuple (bits_c_fc, bits_c_proj).
    finetune: additionally run the STE fine-tune. This is now a JOINT fine-tune (see below), not
        two independent per-block fine-tunes.
    layer_indices: restrict fine-tuning to these layers; the rest still get the GPTQ warm start.
    return_notes: if True, also return ((qkv_notes, mlp_notes)) where each is
        {layer_idx: (bits, committed)}. Under the joint fine-tune both blocks of a tuned layer
        necessarily carry the SAME `committed` flag -- there is one decision per layer now.
    collect_metrics: if True, ALSO return a dict {
        "quant_error": {layer_idx: {"c_attn": ..., "c_fc": ..., "c_proj": ...}},
        "attn_recon": {dataset: {layer_idx: ...}},   # QKV block, merged pre-c_proj attn output
        "mlp_recon":  {dataset: {layer_idx: ...}},   # MLP block output
    }

    JOINT FINE-TUNE (replaces the previous independent per-block fine-tunes)
    -----------------------------------------------------------------------
    Previously each layer ran two separate fine-tunes: QKV against its own MSE+KL attention loss
    with its own Adam and its own commit/reject decision, then MLP against its own MSE loss with
    a second Adam and a second commit/reject decision. Each block was therefore optimized toward
    its own isolated reconstruction target, blind to how its error interacts with the other
    block's error through the residual stream.

    Now, per layer, whichever of {QKV, MLP} is being quantized has its (weight, scale)
    parameters placed in ONE shared Adam -- separate param groups, so each block keeps its own
    LSQ/weight learning rates, but stepped together on every iteration -- against ONE combined
    `block_loss`:

        block_loss = MSE(student block output, float-teacher block output)
                     + lambda_kl * KL(teacher attention maps || student attention maps)

    where "block output" is the whole GPT-2 block's final hidden state (ln_1 -> attention ->
    c_proj -> residual -> ln_2 -> MLP -> residual), computed by `block_forward_flat` for the
    same block input the teacher sees. The KL term is kept as an additive piece of `block_loss`,
    but is only computed when QKV is one of the blocks being tuned in this layer -- it is
    meaningless for an MLP-only tune, where the attention maps are fixed by definition.

    If only one block is being quantized in a layer, the other block still participates in the
    same joint forward with its weights held fixed, so the loss always reflects the true
    end-to-end block output rather than a half-block approximation.

    One commit/reject decision per layer, on one held-out-batch `block_loss`: both tuned blocks'
    weights are written back together, or neither is and both keep their GPTQ warm start.

    Note on cost: `steps_per_block` is now the number of JOINT steps per layer (previously it
    was that many steps for QKV *plus* that many for MLP), so a fine-tuning pass does about half
    the optimizer steps it used to at the same setting.
    """
    n_hessian_batches = HESSIAN_N_BATCHES if n_hessian_batches is None else n_hessian_batches
    group_size = GROUP_SIZE if group_size is None else group_size
    lambda_kl = LAMBDA_KL if lambda_kl is None else lambda_kl
    steps_per_block = JOINT_STEPS_PER_BLOCK if steps_per_block is None else steps_per_block
    lr = JOINT_LR if lr is None else lr
    grad_clip_norm = JOINT_GRAD_CLIP if grad_clip_norm is None else grad_clip_norm
    tune = None if layer_indices is None else set(layer_indices)

    print(f"\n=== pass: {label} ===")
    reset_vram_peak()
    mask = torch.ones(calib_ids[0].shape[1], calib_ids[0].shape[1],
                      dtype=torch.bool, device=DEVICE).tril()

    notes_qkv = {}
    notes_mlp = {}
    quant_errors = {}
    attn_recon = {ds: {} for ds in (metric_eval_chunks or {})} if collect_metrics else {}
    mlp_recon = {ds: {} for ds in (metric_eval_chunks or {})} if collect_metrics else {}

    for i in range(cfg_attn.n_layers):
        layer = load_decoder_layer(reader, config, i)
        act_fn = layer.mlp.act

        # Which blocks take part in THIS layer's joint fine-tune.
        do_ft = finetune and (tune is None or i in tune)
        ft_qkv = do_ft and bits_fn_qkv is not None
        ft_mlp = do_ft and bits_fn_mlp is not None
        joint_ft = ft_qkv or ft_mlp

        # ------------------------------------------------------------------
        # Snapshot float weights for THIS layer, BEFORE either block is quantized.
        # The joint teacher needs BOTH blocks' float weights even when only one of them is
        # being quantized -- the teacher is the whole float block, not a half-float hybrid.
        # ------------------------------------------------------------------
        need_qkv_snapshot = joint_ft or (bits_fn_qkv is not None and collect_metrics)
        need_mlp_snapshot = joint_ft or (bits_fn_mlp is not None and collect_metrics)

        W_qkv_float32 = tuple(W.clone() for W in qkv_weights_io(layer, dtype=torch.float32)) \
            if need_qkv_snapshot else None
        b_qkv = qkv_bias_io(layer, dtype=torch.float32) if need_qkv_snapshot else None
        W_c_attn_float = (layer.attn.c_attn.weight.data.clone().to(torch.float32)
                          if (bits_fn_qkv is not None and collect_metrics) else None)

        W_mlp_float32 = tuple(W.clone() for W in mlp_weights_io(layer, dtype=torch.float32)) \
            if need_mlp_snapshot else None
        b_mlp = mlp_bias_io(layer, dtype=torch.float32) if need_mlp_snapshot else None
        W_c_fc_float = (layer.mlp.c_fc.weight.data.clone().to(torch.float32)
                        if (bits_fn_mlp is not None and collect_metrics) else None)
        W_c_proj_float = (layer.mlp.c_proj.weight.data.clone().to(torch.float32)
                          if (bits_fn_mlp is not None and collect_metrics) else None)

        # ------------------------------------------------------------------
        # (A) QKV block: GPTQ warm start on the fused c_attn matrix.
        #     (The STE fine-tune itself moved to (C) -- it is joint now.)
        # ------------------------------------------------------------------
        qkv_note = ""
        qkv_committed = None
        bits_qkv = None
        ft_qkv_state = None
        if bits_fn_qkv is not None:
            bits_qkv = bits_fn_qkv(i)                    # int, or (bits_Q, bits_K, bits_V)
            bits_qkv3 = _expand_bits(bits_qkv, 3)
            H_qkv = collect_hessian_via_hook(reader, layer.attn.c_attn,
                                             calib_ids[:n_hessian_batches], DEVICE)
            want_scales_qkv = ft_qkv
            out_qkv = quantize_qkv(qkv_of(layer), H_qkv, bits_qkv3, group_size=group_size,
                                   return_scales=want_scales_qkv,
                                   return_compact=want_scales_qkv)
            free(H_qkv)
            qkv_note = _fmt_bits(bits_qkv3, "QKV")

            if want_scales_qkv:
                ws, scales, scale_compact_qkv, g_idx_qkv = out_qkv
                w_flat_qkv = flatten_weights(*ws).clone().requires_grad_(True)
                gptq_w_qkv = w_flat_qkv.detach().clone()
                scale_flat_qkv = flatten_weights(*scales).to(w_flat_qkv.dtype)
                scale_param_qkv = scale_compact_qkv.clone().requires_grad_(True)
                free(ws, scales)
                ft_qkv_state = {
                    "w": w_flat_qkv, "gptq_w": gptq_w_qkv, "scale_flat": scale_flat_qkv,
                    "scale_param": scale_param_qkv, "g_idx": g_idx_qkv,
                    # per-WEIGHT widths, so Q/K/V can sit on different grids in one flat tensor
                    "bits": _bits_per_weight(bits_qkv3, (cfg_attn.q_numel, cfg_attn.kv_numel,
                                                         cfg_attn.kv_numel), w_flat_qkv),
                }
            elif finetune:
                qkv_note += ", GPTQ-only (not in layer_indices)"

        # ------------------------------------------------------------------
        # (B) MLP block: GPTQ warm start on c_fc then c_proj.
        # ------------------------------------------------------------------
        mlp_note = ""
        mlp_committed = None
        bits_mlp = None
        ft_mlp_state = None
        if bits_fn_mlp is not None:
            bits_mlp = bits_fn_mlp(i)                    # int, or (bits_c_fc, bits_c_proj)
            bits_mlp2 = _expand_bits(bits_mlp, 2)
            # c_fc FIRST (Hessian from its own, still-float-upstream input), THEN c_proj, whose
            # Hessian is collected via a FRESH forward pass through `reader` AFTER c_fc is already
            # quantized in place: c_proj's input is act_fn(c_fc(x)), which depends on c_fc's
            # POST-quantization output.
            H_fc = collect_hessian_via_hook(reader, layer.mlp.c_fc,
                                            calib_ids[:n_hessian_batches], DEVICE)
            want_scales_mlp = ft_mlp
            out_fc = gptq_quantize_conv1d(layer.mlp.c_fc, H_fc, bits=bits_mlp2[0],
                                          group_size=group_size, act_order=True,
                                          return_scale=want_scales_mlp,
                                          return_compact=want_scales_mlp)
            free(H_fc)

            H_proj = collect_hessian_via_hook(reader, layer.mlp.c_proj,
                                              calib_ids[:n_hessian_batches], DEVICE)
            out_proj = gptq_quantize_conv1d(layer.mlp.c_proj, H_proj, bits=bits_mlp2[1],
                                            group_size=group_size, act_order=True,
                                            return_scale=want_scales_mlp,
                                            return_compact=want_scales_mlp)
            free(H_proj)
            mlp_note = _fmt_bits(bits_mlp2, ("fc", "proj"))

            if want_scales_mlp:
                W_fc0, scale_fc, scale_compact_fc, g_idx_fc = out_fc
                W_proj0, scale_proj, scale_compact_proj, g_idx_proj = out_proj
                w_flat_mlp = flatten_mlp_weights(W_fc0, W_proj0).clone().requires_grad_(True)
                gptq_w_mlp = w_flat_mlp.detach().clone()
                scale_flat_mlp = flatten_mlp_weights(scale_fc, scale_proj).to(w_flat_mlp.dtype)
                scale_param_fc = scale_compact_fc.clone().requires_grad_(True)
                scale_param_proj = scale_compact_proj.clone().requires_grad_(True)
                free(W_fc0, W_proj0, scale_fc, scale_proj)
                ft_mlp_state = {
                    "w": w_flat_mlp, "gptq_w": gptq_w_mlp, "scale_flat": scale_flat_mlp,
                    "scale_fc": scale_param_fc, "scale_proj": scale_param_proj,
                    "g_idx_fc": g_idx_fc, "g_idx_proj": g_idx_proj,
                    "bits": _bits_per_weight(bits_mlp2, (cfg_mlp.fc_numel, cfg_mlp.proj_numel),
                                             w_flat_mlp),
                }
            elif finetune:
                mlp_note += ", GPTQ-only (not in layer_indices)"

        # ------------------------------------------------------------------
        # (C) JOINT STE fine-tune of whichever blocks were quantized in this layer.
        #     One optimizer (separate param groups), one block_loss, one commit decision.
        # ------------------------------------------------------------------
        if joint_ft:
            n_weights = group_size
            # KL on attention maps only makes sense when QKV is actually being tuned; with QKV
            # fixed the student's attention maps are identical to the teacher's by construction
            # and the term is a constant zero.
            use_kl = bool(ft_qkv_state is not None and lambda_kl)

            # --- one optimizer, separate param groups (weights vs. LSQ scales, per block) ---
            param_groups = []
            all_params = []
            if ft_qkv_state is not None:
                s_mean_qkv = ft_qkv_state["scale_flat"].mean().item()
                param_groups.append({"params": [ft_qkv_state["w"]],
                                     "lr": ALPHA_W * s_mean_qkv})
                param_groups.append({"params": [ft_qkv_state["scale_param"]],
                                     "lr": ALPHA_S * s_mean_qkv})
                all_params += [ft_qkv_state["w"], ft_qkv_state["scale_param"]]
            if ft_mlp_state is not None:
                s_mean_mlp = ft_mlp_state["scale_flat"].mean().item()
                param_groups.append({"params": [ft_mlp_state["w"]],
                                     "lr": ALPHA_W * s_mean_mlp})
                param_groups.append({"params": [ft_mlp_state["scale_fc"],
                                                ft_mlp_state["scale_proj"]],
                                     "lr": ALPHA_S * s_mean_mlp})
                all_params += [ft_mlp_state["w"], ft_mlp_state["scale_fc"],
                               ft_mlp_state["scale_proj"]]
            opt = torch.optim.Adam(param_groups)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps_per_block)

            held_out = len(calib_ids) - 1
            pool = list(range(len(calib_ids) - 1))
            start = (i * steps_per_block) % len(pool)
            order = [pool[(start + s) % len(pool)] for s in range(steps_per_block)]

            def targets_block(j):
                """Block INPUT (the tensor entering ln_1) plus the float teacher's block output
                and attention maps for that same input."""
                h_in = capture_module_input(reader, layer.ln_1, calib_ids[j], DEVICE)
                with torch.no_grad():
                    t_out, t_attn = block_forward_flat(
                        layer, h_in, cfg_attn, cfg_mlp, mask,
                        w_qkv_io=W_qkv_float32, w_mlp_io=W_mlp_float32,
                        b_qkv=b_qkv, b_mlp=b_mlp, act_fn=act_fn, return_attn=use_kl)
                return h_in, t_out, t_attn

            batch_cache = {j: targets_block(j) for j in set(order) | {held_out}}

            def ste_qkv_flat(w, s):
                return ste_quantize(w, expand_scale_qkv(s, ft_qkv_state["g_idx"], cfg_attn),
                                    ft_qkv_state["bits"], n_weights)

            def ste_mlp_flat(w, sfc, sproj):
                return ste_quantize(
                    w, expand_scale_mlp(sfc, ft_mlp_state["g_idx_fc"],
                                        sproj, ft_mlp_state["g_idx_proj"], cfg_mlp),
                    ft_mlp_state["bits"], n_weights)

            def block_loss(j, detach=False):
                """MSE on the whole block's output hidden state + lambda_kl * KL on attention
                maps (the latter only when QKV is one of the tuned blocks). Blocks that are not
                being tuned keep their current weights and still run in this same forward."""
                h_in, t_out, t_attn = batch_cache[j]
                wq_flat = None
                if ft_qkv_state is not None:
                    w = ft_qkv_state["w"]
                    s = ft_qkv_state["scale_param"]
                    wq_flat = ste_qkv_flat(w.detach() if detach else w,
                                           s.detach() if detach else s)
                wm_flat = None
                if ft_mlp_state is not None:
                    w = ft_mlp_state["w"]
                    sfc, sproj = ft_mlp_state["scale_fc"], ft_mlp_state["scale_proj"]
                    wm_flat = ste_mlp_flat(w.detach() if detach else w,
                                           sfc.detach() if detach else sfc,
                                           sproj.detach() if detach else sproj)
                out, attn_w = block_forward_flat(
                    layer, h_in, cfg_attn, cfg_mlp, mask,
                    w_qkv_flat=wq_flat, w_mlp_flat=wm_flat,
                    b_qkv=b_qkv, b_mlp=b_mlp, act_fn=act_fn, return_attn=use_kl)
                mse = F.mse_loss(out, t_out)
                loss = mse
                kl = None
                if use_kl and attn_w is not None and t_attn is not None:
                    kl = F.kl_div(torch.log(attn_w + 1e-8), t_attn, reduction="batchmean")
                    loss = loss + lambda_kl * kl
                if LOG_LOSS_COMPONENTS and not detach:
                    kl_str = (f", kl={kl.item():.6f}, lambda_kl*kl={lambda_kl * kl.item():.6f}"
                              if kl is not None else ", kl=n/a")
                    print(f"    [block_loss] mse={mse.item():.6f}{kl_str}")
                return loss

            def score_block():
                with torch.no_grad():
                    return block_loss(held_out, detach=True).item()

            init_loss = score_block()
            best_loss = init_loss
            best_w_qkv = (ft_qkv_state["w"].detach().clone()
                          if ft_qkv_state is not None else None)
            best_w_mlp = (ft_mlp_state["w"].detach().clone()
                          if ft_mlp_state is not None else None)

            for step_idx, j in enumerate(order):
                opt.zero_grad()
                loss = block_loss(j)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(all_params, grad_clip_norm)
                opt.step()          # both blocks' param groups step together, every iteration
                sched.step()
                if step_idx == steps_per_block - 1 or (step_idx + 1) % SCORE_EVERY == 0:
                    step_loss = score_block()
                    if step_loss < best_loss:
                        best_loss = step_loss
                        with torch.no_grad():
                            if ft_qkv_state is not None:
                                best_w_qkv = ste_qkv_flat(
                                    ft_qkv_state["w"], ft_qkv_state["scale_param"]
                                ).detach().clone()
                            if ft_mlp_state is not None:
                                best_w_mlp = ste_mlp_flat(
                                    ft_mlp_state["w"], ft_mlp_state["scale_fc"],
                                    ft_mlp_state["scale_proj"]).detach().clone()
                free(loss)
            del batch_cache
            free()

            # --- ONE commit/reject decision, shared by both blocks ---
            committed = best_loss <= init_loss
            tuned_names = []
            if ft_qkv_state is not None:
                tuned_names.append("QKV")
            if ft_mlp_state is not None:
                tuned_names.append("MLP")
            joint_tag = "+".join(tuned_names)

            with torch.no_grad():
                flip_qkv = flip_mlp = None
                if ft_qkv_state is not None:
                    sf = ft_qkv_state["scale_flat"]
                    flip_qkv = (torch.round(ft_qkv_state["gptq_w"] / sf) !=
                                torch.round(best_w_qkv / sf)).float().mean().item()
                if ft_mlp_state is not None:
                    sf = ft_mlp_state["scale_flat"]
                    flip_mlp = (torch.round(ft_mlp_state["gptq_w"] / sf) !=
                                torch.round(best_w_mlp / sf)).float().mean().item()

            if committed:
                if ft_qkv_state is not None:
                    write_qkv_flat(layer, best_w_qkv, cfg_attn)
                if ft_mlp_state is not None:
                    write_mlp_flat(layer, best_w_mlp, cfg_mlp)
                verdict = f"joint[{joint_tag}] FT {init_loss:.6f} -> {best_loss:.6f}"
            else:
                verdict = (f"joint[{joint_tag}] kept GPTQ-only "
                           f"({init_loss:.6f} vs FT {best_loss:.6f})")

            if ft_qkv_state is not None:
                qkv_note += f", {verdict}, flip rate {flip_qkv:.3f}"
                qkv_committed = committed
            if ft_mlp_state is not None:
                mlp_note += f", {verdict}, flip rate {flip_mlp:.3f}"
                mlp_committed = committed

            del opt, sched, param_groups, all_params
            del best_w_qkv, best_w_mlp
            del targets_block, block_loss, score_block, ste_qkv_flat, ste_mlp_flat
            # Drop the section (A)/(B) locals too -- otherwise this layer's weight, scale and
            # GPTQ-reference tensors stay alive while the NEXT layer collects its Hessians.
            if ft_qkv_state is not None:
                del w_flat_qkv, gptq_w_qkv, scale_flat_qkv, scale_param_qkv
            if ft_mlp_state is not None:
                del w_flat_mlp, gptq_w_mlp, scale_flat_mlp, scale_param_fc, scale_param_proj
            ft_qkv_state = ft_mlp_state = None
            free()

        if bits_fn_qkv is not None:
            notes_qkv[i] = (bits_qkv, qkv_committed)
        if bits_fn_mlp is not None:
            notes_mlp[i] = (bits_mlp, mlp_committed)

        # ------------------------------------------------------------------
        # (D) Metrics: quant error + reconstruction error, AFTER both blocks are final.
        #     Unchanged -- still measured per block, which is exactly what makes the joint
        #     fine-tune's effect on each block visible separately.
        # ------------------------------------------------------------------
        if collect_metrics:
            layer_errors = {}
            if bits_fn_qkv is not None:
                W_q_full = layer.attn.c_attn.weight.data.to(torch.float32)
                layer_errors["c_attn"] = ((W_q_full - W_c_attn_float).norm()
                                          / W_c_attn_float.norm()).item()
                del W_q_full
            if bits_fn_mlp is not None:
                W_q_fc = layer.mlp.c_fc.weight.data.to(torch.float32)
                W_q_proj = layer.mlp.c_proj.weight.data.to(torch.float32)
                layer_errors["c_fc"] = ((W_q_fc - W_c_fc_float).norm()
                                        / W_c_fc_float.norm()).item()
                layer_errors["c_proj"] = ((W_q_proj - W_c_proj_float).norm()
                                          / W_c_proj_float.norm()).item()
                del W_q_fc, W_q_proj
            quant_errors[i] = layer_errors

            if metric_eval_chunks:
                # QKV block attention-map reconstruction
                if bits_fn_qkv is not None:
                    W_q32_qkv = qkv_weights_io(layer, dtype=torch.float32)
                    for ds_name, chunks in metric_eval_chunks.items():
                        errs = []
                        for ch in chunks:
                            X = capture_module_input(reader, layer.attn.c_attn, ch, DEVICE)
                            with torch.no_grad():
                                A_teacher, _ = compute_attention(*W_qkv_float32, X, cfg_attn,
                                                                 b_Q=b_qkv[0], b_K=b_qkv[1],
                                                                 b_V=b_qkv[2], attn_mask=mask)
                                A_student, _ = compute_attention(*W_q32_qkv, X, cfg_attn,
                                                                 b_Q=b_qkv[0], b_K=b_qkv[1],
                                                                 b_V=b_qkv[2], attn_mask=mask)
                            errs.append(((A_student - A_teacher).norm()
                                        / A_teacher.norm()).item())
                            free(X, A_teacher, A_student)
                        attn_recon[ds_name][i] = sum(errs) / len(errs)
                    del W_q32_qkv

                # MLP block output reconstruction
                if bits_fn_mlp is not None:
                    W_q32_mlp = mlp_weights_io(layer, dtype=torch.float32)
                    for ds_name, chunks in metric_eval_chunks.items():
                        errs = []
                        for ch in chunks:
                            X = capture_module_input(reader, layer.mlp.c_fc, ch, DEVICE)
                            with torch.no_grad():
                                out_teacher = compute_mlp(*W_mlp_float32, X, cfg_mlp, b_fc=b_mlp[0],
                                                         b_proj=b_mlp[1], act_fn=act_fn)
                                out_student = compute_mlp(*W_q32_mlp, X, cfg_mlp, b_fc=b_mlp[0],
                                                         b_proj=b_mlp[1], act_fn=act_fn)
                            errs.append(((out_student - out_teacher).norm()
                                        / out_teacher.norm()).item())
                            free(X, out_teacher, out_student)
                        mlp_recon[ds_name][i] = sum(errs) / len(errs)
                    del W_q32_mlp

        del W_qkv_float32, b_qkv, W_c_attn_float
        del W_mlp_float32, b_mlp, W_c_fc_float, W_c_proj_float
        del layer
        free()

        if verbose:
            parts = []
            if bits_fn_qkv is not None:
                parts.append(f"QKV {qkv_note}")
            if bits_fn_mlp is not None:
                parts.append(f"MLP {mlp_note}")
            print(f"  layer {i:>2}/{cfg_attn.n_layers}: " + (" | ".join(parts) or "fp32 (no quantization)"))

    free(mask)
    ppl = eval_fn()
    vram(f"after '{label}'")
    print(f"=== {label}: perplexity {ppl:.3f} ===")

    notes = (notes_qkv, notes_mlp)
    if collect_metrics:
        metrics = {"quant_error": quant_errors, "attn_recon": attn_recon, "mlp_recon": mlp_recon}
        return (ppl, notes, metrics) if return_notes else (ppl, metrics)
    return (ppl, notes) if return_notes else ppl


print("Validation 7c (deferred from section 7 -- needs pass_quantize_eval, defined just above): "
      "adaptive allocation + JOINT fine-tuning, mixed 2/3/4/8-bit per block (tiny CPU model)...")
_validate_adaptive_joint_pass()
print("PASSED -- pass_quantize_eval(bits_fn_qkv=..., bits_fn_mlp=..., finetune=True) is "
      "bit-width-agnostic end to end for both blocks, under the joint block_loss.")


Validation 7c (deferred from section 7 -- needs pass_quantize_eval, defined just above): adaptive allocation + JOINT fine-tuning, mixed 2/3/4/8-bit per block (tiny CPU model)...

=== pass: 7c: mixed-bit adaptive+joint (tiny CPU model, QKV+MLP) ===
    [vram after '7c: mixed-bit adaptive+joint (tiny CPU model, QKV+MLP)': 0.00 GB now, 0.00 GB peak this pass]
=== 7c: mixed-bit adaptive+joint (tiny CPU model, QKV+MLP): perplexity 257.797 ===
  mixed-bit adaptive+joint pass ran end-to-end on a tiny CPU GPT-2 model (ppl=257.797)
    QKV: L0=2b/FT, L1=3b/FT, L2=4b/FT, L3=8b/FT
    MLP: L0=4b/FT, L1=2b/FT, L2=3b/FT, L3=8b/FT
PASSED -- pass_quantize_eval(bits_fn_qkv=..., bits_fn_mlp=..., finetune=True) is bit-width-agnostic end to end for both blocks, under the joint block_loss.


In [14]:
def _validate_per_matrix_bits(verbose=True):
    """
    7d: per-MATRIX bit-widths (Q, K, V, c_fc, c_proj each their own width). Tiny random GPT-2,
    CPU only, seconds.
      (i)   per-slice GPTQ with equal widths == the fused c_attn GPTQ (so uniform results can't move)
      (ii)  _matrix_trace returns each block's exact trace (diagonal-Hessian toy loss)
      (iii) pass_score emits 5 finite tables per layer
      (iv)  pass_quantize_eval with tuple widths puts every matrix on ITS OWN grid, with and
            without the joint fine-tune (exercises the per-weight-bits STE path)
    """
    from transformers import GPT2Config, GPT2LMHeadModel

    old_device, old_dtype = globals()["DEVICE"], globals()["GPU_DTYPE"]
    globals()["DEVICE"], globals()["GPU_DTYPE"] = "cpu", torch.float32
    try:
        # (i) ---------------------------------------------------------------------------
        torch.manual_seed(3)
        d_in = 32

        class _FakeAttn:
            def __init__(self):
                self.weight = nn.Parameter(torch.randn(d_in, 3 * d_in) * 0.05)

        Xh = torch.randn(400, d_in)
        Hh = (2.0 * Xh.T @ Xh / Xh.shape[0]).double()
        fused, sliced = _FakeAttn(), _FakeAttn()
        sliced.weight.data.copy_(fused.weight.data)
        quantize_qkv(fused, Hh, 4, group_size=8)                          # fused path
        _quantize_qkv_per_slice(sliced, Hh, (4, 4, 4), group_size=8)      # per-slice path
        rel = ((fused.weight.data - sliced.weight.data).norm() / fused.weight.data.norm()).item()
        assert rel < 1e-6, f"per-slice GPTQ != fused GPTQ at equal widths (rel err {rel:.3e})"
        if verbose:
            print(f"  7d(i): per-slice GPTQ == fused GPTQ at equal widths (rel err {rel:.2e})")

        # (ii) --------------------------------------------------------------------------
        d = torch.rand(50) + 0.5
        w0 = torch.randn(50)
        toy_loss = lambda w: 0.5 * (d * w * w).sum()      # diagonal Hessian -> Hutchinson is exact
        for s0, e0 in [(0, 10), (10, 35), (35, 50)]:
            t = _matrix_trace(toy_loss, w0, s0, e0, samples=3)
            want = d[s0:e0].sum().item()
            assert abs(t - want) < 1e-4 * want, f"block [{s0}:{e0}] trace {t} != {want}"
        if verbose:
            print("  7d(ii): _matrix_trace recovers each block's exact trace")

        # (iii) -------------------------------------------------------------------------
        def _tiny_model(seed):
            torch.manual_seed(seed)
            conf = GPT2Config(vocab_size=256, n_embd=32, n_head=4, n_layer=2, n_positions=64,
                              bos_token_id=0, eos_token_id=0)
            m = GPT2LMHeadModel(conf)
            m.config._attn_implementation = "eager"
            m.eval()
            for p in m.parameters():
                p.requires_grad_(False)
            return m

        ref = _tiny_model(4)
        cfg, mlp_cfg = AttnConfig(ref.config), MLPConfig(ref.config)
        calib = [torch.randint(0, 256, (1, 16)) for _ in range(4)]
        scores, alloc = pass_score(ref, ref.config, cfg, mlp_cfg, calib, samples=2,
                                   n_score_batches=1, n_hessian_batches=2, group_size=8,
                                   verbose=False)
        expect = {unit_name(i, m) for i in range(cfg.n_layers) for m in MATRICES}
        assert set(scores) == set(alloc) == expect, "pass_score must emit 5 units per layer"
        for n, tbl in alloc.items():
            assert set(tbl) == set(BIT_WIDTHS) and all(math.isfinite(v) for v in tbl.values()), n
        assert all(math.isfinite(v) for v in scores.values())
        if verbose:
            print(f"  7d(iii): pass_score -> {len(scores)} units ({len(MATRICES)} per layer), "
                  "all tables finite")
        free(ref)

        # (iv) --------------------------------------------------------------------------
        eval_ids = torch.randint(0, 256, (48,))
        BITS_QKV, BITS_MLP = (2, 3, 8), (2, 8)
        n_groups = d_in // 8

        def _max_unique(W):       # most distinct values in any single output column
            return max(torch.unique(W[:, c]).numel() for c in range(W.shape[1]))

        for finetune in (False, True):
            model = _tiny_model(5)

            def synth_eval(model=model):
                nll, n = _sliding_window_nll(model, eval_ids, max_length=16, stride=16)
                return math.exp(nll / n)

            ppl, notes = pass_quantize_eval(
                model, model.config, cfg, mlp_cfg, calib, synth_eval,
                bits_fn_qkv=lambda i: BITS_QKV, bits_fn_mlp=lambda i: BITS_MLP,
                finetune=finetune, steps_per_block=2, n_hessian_batches=2, group_size=8,
                label=f"7d: per-matrix widths QKV={BITS_QKV} MLP={BITS_MLP} finetune={finetune}",
                verbose=False, return_notes=True)
            assert math.isfinite(ppl) and ppl > 0
            assert notes[0][0][0] == BITS_QKV and notes[1][0][0] == BITS_MLP, "notes lost the widths"

            layer0 = model.transformer.h[0]
            W_Q, W_K, W_V = qkv_weights_io(layer0)
            W_fc, W_proj = mlp_weights_io(layer0)
            uq, uk, uv = _max_unique(W_Q), _max_unique(W_K), _max_unique(W_V)
            # symmetric grid: 2^b - 1 levels per (column, group) -> at most n_groups * (2^b - 1)
            assert uq <= n_groups * (2 ** 2 - 1), f"Q not on a 2-bit grid ({uq} distinct values)"
            assert uk <= n_groups * (2 ** 3 - 1), f"K not on a 3-bit grid ({uk} distinct values)"
            assert uv > n_groups * (2 ** 2 - 1), f"V looks 2-bit ({uv} distinct values)"
            assert _max_unique(W_fc) <= n_groups * (2 ** 2 - 1), "c_fc not on a 2-bit grid"
            assert _max_unique(W_proj) > n_groups * (2 ** 2 - 1), "c_proj looks 2-bit"
            if verbose:
                print(f"  7d(iv) finetune={finetune}: distinct values per column  Q={uq}  K={uk}  "
                      f"V={uv} | c_fc={_max_unique(W_fc)}  c_proj={_max_unique(W_proj)}  (ppl {ppl:.2f})")
            free(model)
    finally:
        globals()["DEVICE"], globals()["GPU_DTYPE"] = old_device, old_dtype


print("Validation 7d: per-matrix (Q/K/V/c_fc/c_proj) bit-widths + per-matrix Hessian traces "
      "(tiny CPU model)...")
_validate_per_matrix_bits()
print("PASSED -- every matrix is scored on its own and quantized on its own grid.")


Validation 7d: per-matrix (Q/K/V/c_fc/c_proj) bit-widths + per-matrix Hessian traces (tiny CPU model)...
  7d(i): per-slice GPTQ == fused GPTQ at equal widths (rel err 0.00e+00)
  7d(ii): _matrix_trace recovers each block's exact trace
Scoring 2 layers x 5 matrices (Q,K,V,c_fc,c_proj): 2 Hutchinson probes x 1 batches per matrix, Q/K/V loss = MSE + 0.1*KL, c_fc/c_proj loss = MSE, perturbation mode 'rtn'
    [vram after scoring pass: 0.00 GB now, 0.00 GB peak this pass]
  7d(iii): pass_score -> 10 units (5 per layer), all tables finite

=== pass: 7d: per-matrix widths QKV=(2, 3, 8) MLP=(2, 8) finetune=False ===
    [vram after '7d: per-matrix widths QKV=(2, 3, 8) MLP=(2, 8) finetune=False': 0.00 GB now, 0.00 GB peak this pass]
=== 7d: per-matrix widths QKV=(2, 3, 8) MLP=(2, 8) finetune=False: perplexity 259.356 ===
  7d(iv) finetune=False: distinct values per column  Q=9  K=22  V=32 | c_fc=9  c_proj=128  (ppl 259.36)

=== pass: 7d: per-matrix widths QKV=(2, 3, 8) MLP=(2, 8) finetune=True

## 9. Setup

In [15]:
# --- Resumable checkpointing (runs/<run_id>/results.json) ---
# Each kernel session gets its own timestamped run directory, so restarting the kernel
# and re-running the notebook always starts a clean sweep (no [skip] spam) while every
# previous run's results.json/.csv stay exactly where they were, untouched.
#
# Re-running just THIS cell (or the sweep cell) without restarting the kernel reuses the
# same RUN_ID -- that's what gives you crash-resume/skip-already-done behavior *within* one
# session. Set FORCE_RERUN=True to redo every arm even inside the same run directory.
if "RUN_ID" not in globals():
    RUN_ID = time.strftime("%Y%m%d-%H%M%S")

RUN_DIR = os.path.join("runs", RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(RUN_DIR, "results.json")
print(f"This session's run directory: {RUN_DIR}")

# Convenience pointer to the most recent run -- always overwritten, never read from.
LATEST_LINK = os.path.join("runs", "latest")
try:
    if os.path.islink(LATEST_LINK) or os.path.exists(LATEST_LINK):
        os.remove(LATEST_LINK)
    os.symlink(RUN_ID, LATEST_LINK)
except OSError:
    pass  # symlinks unsupported (e.g. some Windows setups) -- harmless to skip


def load_checkpoint():
    if FORCE_RERUN:
        print(f"FORCE_RERUN=True -- ignoring any existing {CHECKPOINT_PATH} and redoing every arm")
        return [], {}
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH) as f:
            state = json.load(f)
        print(f"Resuming from {CHECKPOINT_PATH}: "
              f"{len(state.get('per_layer_store', {}))} arm(s) already recorded")
        return state.get("results_rows", []), state.get("per_layer_store", {})
    return [], {}


def flush_checkpoint(results_rows, per_layer_store):
    tmp = CHECKPOINT_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump({"results_rows": results_rows, "per_layer_store": per_layer_store},
                  f, indent=2)
    os.replace(tmp, CHECKPOINT_PATH)   # atomic -- a crash mid-write can't corrupt it


This session's run directory: runs/20260924-080723


In [16]:
print(f"Loading {MODEL_ID}'s tokenizer and config (cached after the first run)...")
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
config = AutoConfig.from_pretrained(MODEL_ID)
config._attn_implementation = "eager"

ATTN = AttnConfig(config)
MLP = MLPConfig(config)
print(ATTN)
print(MLP)
assert ATTN.hidden_size % GROUP_SIZE == 0, (
    f"GROUP_SIZE={GROUP_SIZE} must divide hidden_size={ATTN.hidden_size} -- true for gpt2 (768), "
    f"gpt2-medium (1024) and gpt2-large (1280), but NOT gpt2-xl (1600)")
print(f"Q/K/V parameters per layer: {ATTN.qkv_numel:,}")
print(f"MLP parameters per layer (c_fc + c_proj): {MLP.mlp_numel:,}")

print(f"\nBuilding the C4 calibration set (train split, streaming, seed {CALIB_SEED})...")
calib_ids = build_calibration_ids(tokenizer)
print(f"{len(calib_ids)} calibration chunks of {CALIB_SEQ_LEN} tokens")

print(f"\nBuilding eval corpora: WikiText-2 test + C4 validation ({C4_EVAL_N_DOCS} docs, "
      "disjoint from calibration by split)...")
wikitext2_eval_text = load_wikitext2_eval_text()
c4_eval_text = load_c4_eval_text()
EVAL_TEXTS = {"wikitext2": wikitext2_eval_text, "c4": c4_eval_text}
print(f"Evaluation: max_length={EVAL_MAX_LENGTH}, stride={EVAL_STRIDE}, "
      f"scored on both {list(EVAL_TEXTS)}.")

print(f"\nBuilding {ATTN_RECON_N_CHUNKS} reconstruction-metric eval chunks per dataset...")
metric_eval_chunks = {
    "wikitext2": build_metric_eval_chunks(tokenizer, wikitext2_eval_text, ATTN_RECON_N_CHUNKS,
                                          CALIB_SEQ_LEN, seed=CALIB_SEED + 1),
    "c4": build_metric_eval_chunks(tokenizer, c4_eval_text, ATTN_RECON_N_CHUNKS,
                                   CALIB_SEQ_LEN, seed=CALIB_SEED + 2),
}

results_rows, per_layer_store = load_checkpoint()


def eval_all(reader):
    """Perplexity + top-1 accuracy on both eval datasets, one forward pass each."""
    return {ds: evaluate_perplexity_and_accuracy(reader, tokenizer, text,
                                                 max_length=EVAL_MAX_LENGTH, stride=EVAL_STRIDE)
           for ds, text in EVAL_TEXTS.items()}


Loading gpt2's tokenizer and config (cached after the first run)...
AttnConfig(hidden=768, layers=12, heads=12, head_dim=64)
MLPConfig(hidden=768, intermediate=3072, layers=12, act=gelu_new)
Q/K/V parameters per layer: 1,769,472
MLP parameters per layer (c_fc + c_proj): 4,718,592

Building the C4 calibration set (train split, streaming, seed 42)...


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1807 > 1024). Running this sequence through the model will result in indexing errors


128 calibration chunks of 512 tokens

Building eval corpora: WikiText-2 test + C4 validation (2000 docs, disjoint from calibration by split)...


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Evaluation: max_length=1024, stride=512, scored on both ['wikitext2', 'c4'].

Building 8 reconstruction-metric eval chunks per dataset...


## 10. fp32 control

In [17]:
reader, _ = load_model_and_tokenizer(MODEL_ID, DEVICE)   # fresh, unquantized control
for ds_name, (ppl, acc) in eval_all(reader).items():
    results_rows.append({"mode": "fp32", "avg_bits": None, "eval_dataset": ds_name,
                         "perplexity": ppl, "accuracy": acc,
                         "quant_error_mean": 0.0,
                         "attn_recon_error_mean": 0.0, "mlp_recon_error_mean": 0.0})
    print(f"fp32 control [{ds_name}]: perplexity {ppl:.3f}, accuracy {acc:.4f}")
    if not (15.0 < ppl < 60.0):
        print(f"  WARNING: perplexity {ppl:.3f} is outside the expected ~18-30 range for "
              f"{MODEL_ID} -- investigate the pipeline before reading anything into the sweep.")
del reader
free()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

fp32 control [wikitext2]: perplexity 24.357, accuracy 0.4148
fp32 control [c4]: perplexity 32.770, accuracy 0.3736


## 11. JAB-Hessian scoring (one pass, reused by every budget)

One scoring pass produces a Hessian trace and a sensitivity table for **every matrix** -- `Q`, `K`,
`V`, `FC` (c_fc), `PROJ` (c_proj) -- i.e. 5 * `n_layers` units. The greedy allocator then assigns each
matrix its own bit-width.


In [18]:
reader, _ = load_model_and_tokenizer(MODEL_ID, DEVICE)   # fresh, unquantized -- see pass_score's
torch.manual_seed(42)
jab_scores, allocator_input = pass_score(reader, config, ATTN, MLP, calib_ids)
del reader
free()

N_UNITS = len(allocator_input)
assert N_UNITS == len(MATRICES) * ATTN.n_layers, (
    f"expected {len(MATRICES) * ATTN.n_layers} allocator units (Q,K,V,c_fc,c_proj per layer), "
    f"got {N_UNITS}")


# Real parameter count per allocator unit. The units are no longer equal: Q, K, V hold
# hidden^2 weights each, c_fc and c_proj hold hidden*4*hidden each (4x bigger). Weighting the
# knapsack by real numel makes `avg_bits` a true parameter-weighted average bit-width, which is what
# a memory budget actually is -- a per-unit budget would treat a 590k-weight K like a 2.4M-weight c_fc.
_UNIT_NUMEL = {"Q": ATTN.q_numel, "K": ATTN.kv_numel, "V": ATTN.kv_numel,
               "FC": MLP.fc_numel, "PROJ": MLP.proj_numel}
UNIT_PARAM_COUNTS = {}
for _i in range(ATTN.n_layers):
    for _m in MATRICES:
        UNIT_PARAM_COUNTS[unit_name(_i, _m)] = _UNIT_NUMEL[_m]
TOTAL_UNIT_PARAMS = sum(UNIT_PARAM_COUNTS.values())


def jab_assignment_for(avg_bits):
    """Greedy allocation at a given average-bits budget -- reuses `allocator_input` from the ONE
    scoring pass above; scores don't depend on the budget, so this just re-solves the greedy
    knapsack, not a re-scoring. The budget is over ALL (layer, block) units, weighted by each
    unit's real parameter count so avg_bits is a true parameter-weighted average bit-width."""
    budget = avg_bits * TOTAL_UNIT_PARAMS
    assignment, cost_used, sensitivity = greedy_allocate(allocator_input, budget,
                                                          param_counts=UNIT_PARAM_COUNTS)
    print(f"  avg_bits={avg_bits}: budget {budget:.3e}, used {cost_used:.3e} "
          f"({cost_used / TOTAL_UNIT_PARAMS:.2f} avg), sensitivity {sensitivity:.4e}")
    return assignment


# ---------------------------------------------------------------------------
# Group-level score normalization (QKV vs. MLP), and the allocator built on it
# ---------------------------------------------------------------------------
# The raw `allocator_input` tables are NOT on a shared scale across block types: a QKV unit's
# score comes from the Hessian trace of the joint MSE + lambda_kl*KL attention loss, an MLP
# unit's from the trace of a plain MSE reconstruction loss. Those two losses have different
# units and different typical magnitudes, so pooling them into ONE greedy knapsack -- even
# weighted by real parameter counts, as `jab_assignment_for` does -- makes the comparison
# "QKV sensitivity vs. MLP sensitivity" meaningless. In practice it drove MLP down to 2 bits in
# several layers and blew up perplexity.

def normalize_allocator_input_grouped(allocator_input, ref_bits=None):
    """Group-level scale-free version of allocator_input: divide every unit's per-bit table by
    a SHARED reference computed once per block type (QKV, MLP) -- NOT by that unit's own
    ref_bits value. Per-unit normalization cancels each unit's trace multiplier algebraically,
    since table[unit][b] = trace[unit] * pert[unit](b):
    table[unit][b] / table[unit][ref_bits] = pert[unit](b) / pert[unit](ref_bits), which has NO
    trace term left in it at all. Confirmed empirically: that ratio came out ~1.1-1.2e8 for
    every single unit regardless of layer or block (pure bit-grid geometry from `pert`, since
    the pert(b)/pert(16) shape is nearly identical across units) -- so a per-unit-normalized
    allocation ends up equivalent to flat uniform bit-width, with the JAB trace contributing
    nothing.
    This version divides by one shared per-block-type reference (the mean of that block's
    units' ref_bits scores) instead, which rescales QKV vs. MLP onto comparable magnitudes
    while leaving each unit's own trace differences intact WITHIN its block."""
    ref_bits = max(BIT_WIDTHS) if ref_bits is None else ref_bits
    groups = {}
    for name in allocator_input:
        groups.setdefault(block_of(name), []).append(name)
    group_ref = {
        g: sum(allocator_input[n][ref_bits] for n in names) / len(names)
        for g, names in groups.items()
    }
    normalized = {}
    for name, table in allocator_input.items():
        ref = group_ref[block_of(name)]
        normalized[name] = {b: (v / ref if ref else v) for b, v in table.items()}
    return normalized


# Hard floor for every adaptive allocation in the sweep. 3 keeps the allocator off the 2-bit
# cliff while still leaving 3/4/8/16 of headroom to vary per layer. Set to None to allow 2-bit.
ALLOC_MIN_BITS = 3


def jab_assignment_for_normalized_grouped(avg_bits, min_bits=None, ref_bits=None,
                                          param_counts=UNIT_PARAM_COUNTS):
    """Pooled allocation over `normalize_allocator_input_grouped` -- the corrected version of
    the normalized allocator, preserving within-block trace differences instead of canceling
    them out. This is what section 12 uses.

    param_counts: defaults to UNIT_PARAM_COUNTS, so the budget is parameter-weighted and
      avg_bits reads as a true parameter-weighted average bit-width (the units are now single
      matrices of different sizes, so a per-unit budget would be meaningless). Pass None for the
      old per-unit budget (`avg_bits * N_UNITS`).

    Grouping: units are normalized per LOSS group (`block_of`): the Q/K/V traces come from the
    attention loss, c_fc/c_proj from the MLP loss. Inside a group every matrix has the same size,
    so the traces compare directly and the allocator can tell Q from K from V, layer by layer."""
    normalized_input = normalize_allocator_input_grouped(allocator_input, ref_bits=ref_bits)
    if param_counts is None:
        budget = avg_bits * N_UNITS
        denom = N_UNITS
    else:
        budget = avg_bits * sum(param_counts.values())
        denom = sum(param_counts.values())
    assignment, cost_used, sensitivity = greedy_allocate(normalized_input, budget,
                                                         param_counts=param_counts,
                                                         min_bits=min_bits)
    print(f"  [normalized_grouped] avg_bits={avg_bits}, min_bits={min_bits}: budget "
          f"{budget:.4g}, used {cost_used:.4g} ({cost_used / denom:.2f} avg), "
          f"sensitivity {sensitivity:.4e}")
    # Report the true parameter-weighted average too, since the unit-weighted budget above and
    # the model's real average bit-width are not the same number.
    real_avg = (sum(UNIT_PARAM_COUNTS[n] * b for n, b in assignment.items())
                / TOTAL_UNIT_PARAMS)
    print(f"  [normalized_grouped] real parameter-weighted average: {real_avg:.3f} bits")
    return assignment


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Scoring 12 layers x 5 matrices (Q,K,V,c_fc,c_proj): 10 Hutchinson probes x 2 batches per matrix, Q/K/V loss = MSE + 0.1*KL, c_fc/c_proj loss = MSE, perturbation mode 'rtn'
  layer_0_Q: trace=2288.9739 [HIGH]  2b=4.609e+07, 3b=6.368e+06, 4b=1.172e+06, 8b=3.569e+03, 16b=5.358e-02
  layer_0_K: trace=4043.4268 [HIGH]  2b=8.795e+07, 3b=1.605e+07, 4b=3.040e+06, 8b=9.251e+03, 16b=1.391e-01
  layer_0_V: trace=8.2787 [LOW]  2b=9.959e+03, 3b=1.413e+03, 4b=2.605e+02, 8b=7.906e-01, 16b=1.192e-05
  layer_0_FC: trace=107.0613 [HIGH]  2b=2.966e+06, 3b=4.585e+05, 4b=8.796e+04, 8b=2.743e+02, 16b=4.118e-03
  layer_0_PROJ: trace=232.1027 [HIGH]  2b=2.466e+06, 3b=4.302e+05, 4b=9.160e+04, 8b=3.998e+02, 16b=6.029e-03
  layer 0: Q+K+V trace sum = 6340.6793, c_fc+c_proj trace sum = 339.1640
  layer_1_Q: trace=8855.1060 [HIGH]  2b=7.085e+07, 3b=9.959e+06, 4b=1.914e+06, 8b=5.868e+03, 16b=8.831e-02
  layer_1_K: trace=9405.2124 [HIGH]  2b=8.394e+07, 3b=1.171e+07, 4b=2.171e+06, 8b=6.606e+03, 16b=9.954e-02
  layer_

## 12. Mode x bit-budget sweep

`uniform`/`joint` apply one flat bit-width to every matrix (undefined at
fractional `SWEEP_BITS` -- see `UNIFORM_AVAILABLE_WIDTHS`). `jab`/`adaptive_joint` mix
per-unit bit-widths via `jab_assignment_for_normalized_grouped` (group-normalized scores + an `ALLOC_MIN_BITS` floor) to hit any average budget, so all five
`SWEEP_BITS` entries are defined for them.

`jab`/`adaptive_joint`/`jab_prop` now allocate **per matrix** (Q, K, V, c_fc, c_proj) from each
matrix's own Hessian trace, on a parameter-weighted budget. `type_offset` works per matrix too.


In [19]:
# Type-Offset allocator, ported from run_mistral_quant.py's `type_offset_allocate`.
# NO scoring pass: a fixed per-block-TYPE offset table around one base width, then a
# deterministic per-unit pass that lands the parameter-weighted average on the budget.
TYPE_OFFSETS = {"q": 0, "k": 0, "v": 0, "fc": 0, "proj": 0}   # bits relative to the base width, per
                                      # matrix type. E.g. {"fc": +1, "proj": -1} is the report's
                                      # c_fc = b+1 / c_proj = b-1 pattern.
TYPE_OFFSET_ANY_WIDTH = False         # gap closing moves ONE BIT_WIDTHS grid step at a time
                                      # (what run_mistral_quant.py's docstring says). True lets a
                                      # unit jump to ANY width (what its code actually does).


def _nearest_bits(x, bit_widths):
    """Nearest available bit-width, ties to the LOWER width, so clamping onto the grid never
    silently spends memory it was not asked to."""
    return min(sorted(bit_widths), key=lambda b: (abs(b - x), b))


def type_offset_assignment_for(avg_bits, min_bits=None, offsets=None, any_width=None):
    """Per-unit bit assignment for the `type_offset` mode. Returns
    {layer_{i}_{Q,K,V,FC,PROJ}: bits}, same shape as `jab_assignment_for_normalized_grouped`, so it
    drops straight into `run_arm`.

      1. Scan a continuous base `t` in 0.01 steps; each block type takes
         nearest(t + offsets[type]) on the BIT_WIDTHS grid (at/above the `min_bits` floor).
         Keep the `t` whose parameter-weighted cost is closest to the budget.
      2. Close the remaining gap by moving INDIVIDUAL units, always taking the single move that
         most reduces |cost - budget|. Ties are broken by a fixed order -- larger offset first,
         then layer index, then QKV before MLP -- so the result is deterministic.

    The budget is parameter-weighted (`avg_bits * TOTAL_UNIT_PARAMS`), so avg_bits is a true
    average bit-width per parameter. Nothing here reads `allocator_input`."""
    offsets = dict(TYPE_OFFSETS if offsets is None else {**TYPE_OFFSETS, **offsets})
    any_width = TYPE_OFFSET_ANY_WIDTH if any_width is None else any_width
    floor = min(BIT_WIDTHS) if min_bits is None else min_bits
    widths = sorted(b for b in BIT_WIDTHS if b >= floor)

    kinds = tuple(m.lower() for m in MATRICES)      # q, k, v, fc, proj
    unit_kind, unit_layer = {}, {}
    for i in range(ATTN.n_layers):
        for m in MATRICES:
            unit_kind[unit_name(i, m)], unit_layer[unit_name(i, m)] = m.lower(), i
    names = list(unit_kind)
    params = {n: UNIT_PARAM_COUNTS[n] for n in names}
    total = sum(params.values())
    budget = avg_bits * total

    # --- step 1: continuous base scan ---
    lo = min(widths) - max(offsets.values()) - 1
    hi = max(widths) - min(offsets.values()) + 1
    best = None
    for s in range(int(round((hi - lo) / 0.01)) + 1):
        t = lo + 0.01 * s
        assign = {n: _nearest_bits(t + offsets[unit_kind[n]], widths) for n in names}
        c = sum(cost(assign[n], params[n]) for n in names)
        if best is None or abs(c - budget) < abs(best[0] - budget):
            best = (c, assign)
    cost_used, assign = best[0], dict(best[1])
    base_avg = cost_used / total

    # --- step 2: per-unit gap closing ---
    order = sorted(names, key=lambda n: (-offsets[unit_kind[n]], unit_layer[n],
                                         kinds.index(unit_kind[n])))
    n_moves = 0
    while True:
        gap = abs(cost_used - budget)
        move = None
        for n in order:
            cur = assign[n]
            if any_width:
                options = [b for b in widths if b != cur]
            else:
                k = widths.index(cur)
                options = [widths[j] for j in (k - 1, k + 1) if 0 <= j < len(widths)]
            for nb in options:
                new_cost = cost_used + cost(nb, params[n]) - cost(cur, params[n])
                gain = gap - abs(new_cost - budget)
                if gain > 1e-9 and (move is None or gain > move[0]):
                    move = (gain, n, nb, new_cost)
        if move is None:
            break
        _, n, nb, cost_used = move
        assign[n] = nb
        n_moves += 1

    mean_bits = {k: sum(assign[n] for n in names if unit_kind[n] == k) / ATTN.n_layers
                 for k in kinds}
    print(f"  [type_offset] avg_bits={avg_bits}, min_bits={min_bits}, offsets={offsets}: "
          f"base-only {base_avg:.3f} -> real parameter-weighted average "
          f"{cost_used / total:.3f} bits after {n_moves} unit move(s); "
          "mean bits " + ", ".join(f"{k}={v:.2f}" for k, v in mean_bits.items()))
    return assign


### Error-propagation-weighted allocation 

In [20]:
def apply_propagation_weight(table, n_layers):
    """Depth-weight a {unit_name: {bits: score}} table: multiply every unit's per-bit
    score by (n_layers - layer_idx), so an early layer's quantization error -- which
    propagates through more downstream layers before reaching the model's output -- 
    counts for more in the greedy knapsack than a late layer's. Ported from
    run_mlp_quant.py's apply_propagation_weight; `layer_idx_of` (section 4) parses the
    same "layer_{i}_..." unit-name prefix that script's `layer_idx_of` does."""
    return {n: {b: v * (n_layers - layer_idx_of(n)) for b, v in t.items()}
            for n, t in table.items()}


def jab_prop_assignment_for(avg_bits, min_bits=None, ref_bits=None):
    """Same group-normalized table `jab_assignment_for_normalized_grouped` allocates
    over, with the error-propagation depth weight layered on top before the greedy
    knapsack runs. Returns {layer_{i}_{Q,K,V,FC,PROJ}: bits}, same shape as every other
    assignment function here, so it drops straight into `run_arm`."""
    normalized_input = normalize_allocator_input_grouped(allocator_input, ref_bits=ref_bits)
    weighted_input = apply_propagation_weight(normalized_input, ATTN.n_layers)
    budget = avg_bits * TOTAL_UNIT_PARAMS    # parameter-weighted, same as
                                              # jab_assignment_for_normalized_grouped's default
    assignment, cost_used, sensitivity = greedy_allocate(weighted_input, budget,
                                                         param_counts=UNIT_PARAM_COUNTS,
                                                         min_bits=min_bits)
    print(f"  [jab_prop] avg_bits={avg_bits}, min_bits={min_bits}: budget {budget:.4g}, "
          f"used {cost_used:.4g} ({cost_used / TOTAL_UNIT_PARAMS:.2f} avg), "
          f"sensitivity {sensitivity:.4e}")
    real_avg = (sum(UNIT_PARAM_COUNTS[n] * b for n, b in assignment.items())
                / TOTAL_UNIT_PARAMS)
    print(f"  [jab_prop] real parameter-weighted average: {real_avg:.3f} bits")
    return assignment


def jab_raw_assignment_for(avg_bits, min_bits=None):
    """`jab` WITHOUT group normalization: the greedy knapsack runs directly on the raw
    `allocator_input` (trace * perturbation), so the Q/K/V attention-loss traces and the
    c_fc/c_proj MLP-loss traces compete on their native, unrescaled scales. Everything else
    matches `jab_assignment_for_normalized_grouped` (parameter-weighted budget, same
    `min_bits` floor), so the only difference between the two arms is the normalization.
    Returns {layer_{i}_{Q,K,V,FC,PROJ}: bits}."""
    budget = avg_bits * TOTAL_UNIT_PARAMS
    assignment, cost_used, sensitivity = greedy_allocate(allocator_input, budget,
                                                         param_counts=UNIT_PARAM_COUNTS,
                                                         min_bits=min_bits)
    print(f"  [jab_raw] avg_bits={avg_bits}, min_bits={min_bits}: budget {budget:.4g}, "
          f"used {cost_used:.4g} ({cost_used / TOTAL_UNIT_PARAMS:.2f} avg), "
          f"sensitivity {sensitivity:.4e}")
    real_avg = (sum(UNIT_PARAM_COUNTS[n] * b for n, b in assignment.items())
                / TOTAL_UNIT_PARAMS)
    print(f"  [jab_raw] real parameter-weighted average: {real_avg:.3f} bits")
    return assignment


In [21]:
def run_arm(mode, avg_bits):
    """
    Fresh model load, quantize + evaluate one (mode, avg_bits) arm on both eval datasets.
    Returns (rows, per_layer_metrics), or None if this arm is undefined (a flat-bit mode at a
    fractional budget).
    """
    if mode in ("uniform", "joint") and avg_bits not in UNIFORM_AVAILABLE_WIDTHS:
        return None

    reader, _ = load_model_and_tokenizer(MODEL_ID, DEVICE)   # own fresh model
    finetune = mode in ("joint", "adaptive_joint")

    if mode in ("uniform", "joint"):
        bits_qkv = int(avg_bits)
        bits_mlp = int(avg_bits)
        bits_fn_qkv = lambda i: bits_qkv
        bits_fn_mlp = lambda i: bits_mlp
    elif mode == "type_offset":
        # No scoring: per-block-TYPE offsets around a base width, then per-unit gap closing to
        # land on the parameter-weighted budget -- see the type-offset markdown cell above.
        assignment = type_offset_assignment_for(
            avg_bits, min_bits=ALLOC_MIN_BITS)       # {layer_{i}_{Q,K,V,FC,PROJ}: bits}
        bits_fn_qkv, bits_fn_mlp = bits_fns_from_assignment(assignment)   # per-matrix tuples
    elif mode == "jab_raw":
        # jab without the group normalization: same knapsack, floor and budget, but on the raw
        # trace * perturbation tables (see jab_raw_assignment_for).
        assignment = jab_raw_assignment_for(
            avg_bits, min_bits=ALLOC_MIN_BITS)       # {layer_{i}_{Q,K,V,FC,PROJ}: bits}
        bits_fn_qkv, bits_fn_mlp = bits_fns_from_assignment(assignment)   # per-matrix tuples
    elif mode == "jab_prop":
        # jab's per-layer knapsack again, but with an error-propagation depth weight layered
        # onto the group-normalized scores first -- see the markdown cell above for how this
        # maps onto run_mlp_quant.py's PROPAGATION_WEIGHT / apply_propagation_weight.
        assignment = jab_prop_assignment_for(
            avg_bits, min_bits=ALLOC_MIN_BITS)       # {layer_{i}_{Q,K,V,FC,PROJ}: bits}
        bits_fn_qkv, bits_fn_mlp = bits_fns_from_assignment(assignment)   # per-matrix tuples
    else:
        # Allocation is done ENTIRELY on the group-normalized scores (QKV and MLP each divided
        # by one shared per-block-type reference), with a hard ALLOC_MIN_BITS floor. The old
        # raw-score, parameter-weighted `jab_assignment_for` is still defined in section 9 for
        # A/B comparison, but is no longer what the sweep runs on.
        assignment = jab_assignment_for_normalized_grouped(
            avg_bits, min_bits=ALLOC_MIN_BITS)       # {layer_{i}_{Q,K,V,FC,PROJ}: bits}
        bits_fn_qkv, bits_fn_mlp = bits_fns_from_assignment(assignment)   # per-matrix tuples

    cache = {}

    def eval_fn():
        cache["ds_evals"] = eval_all(reader)
        return cache["ds_evals"]["wikitext2"][0]

    _, notes, metrics = pass_quantize_eval(
        reader, config, ATTN, MLP, calib_ids, eval_fn,
        bits_fn_qkv=bits_fn_qkv, bits_fn_mlp=bits_fn_mlp,
        finetune=finetune, layer_indices=JOINT_LAYERS,
        label=f"{mode} @ {avg_bits} avg bits",
        return_notes=True, collect_metrics=True, metric_eval_chunks=metric_eval_chunks)

    # quant_error is {layer_idx: {"c_attn": ..., "c_fc": ..., "c_proj": ...}} -- flatten all
    # matrices' errors across every layer into one mean.
    quant_flat = [v for layer_errs in metrics["quant_error"].values()
                 for v in layer_errs.values()]
    quant_mean = sum(quant_flat) / len(quant_flat) if quant_flat else float("nan")

    rows = []
    for ds_name, (ppl, acc) in cache["ds_evals"].items():
        attn_recon_ds = metrics["attn_recon"].get(ds_name, {})
        mlp_recon_ds = metrics["mlp_recon"].get(ds_name, {})
        attn_mean = (sum(attn_recon_ds.values()) / len(attn_recon_ds)
                     if attn_recon_ds else float("nan"))
        mlp_mean = (sum(mlp_recon_ds.values()) / len(mlp_recon_ds)
                    if mlp_recon_ds else float("nan"))
        rows.append({"mode": mode, "avg_bits": avg_bits, "eval_dataset": ds_name,
                     "perplexity": ppl, "accuracy": acc,
                     "quant_error_mean": quant_mean,
                     "attn_recon_error_mean": attn_mean,
                     "mlp_recon_error_mean": mlp_mean})

    # Convert the (qkv_notes, mlp_notes) tuple into the JSON-friendly dict we persist.
    qkv_notes, mlp_notes = notes
    per_layer = {
        "quant_error": metrics["quant_error"],
        "attn_recon": metrics["attn_recon"],
        "mlp_recon": metrics["mlp_recon"],
        "notes_qkv": {str(k): v for k, v in qkv_notes.items()},
        "notes_mlp": {str(k): v for k, v in mlp_notes.items()},
    }
    del reader
    free()
    return rows, per_layer


for mode in MODES:
    for avg_bits in SWEEP_BITS:
        key = f"{mode}_{avg_bits}"
        already_done = not FORCE_RERUN and (key in per_layer_store or any(
            r["mode"] == mode and r["avg_bits"] == avg_bits for r in results_rows))
        if already_done:
            print(f"[skip] {mode} @ {avg_bits}: already in {CHECKPOINT_PATH}")
            continue

        out = run_arm(mode, avg_bits)
        if out is None:
            for ds_name in EVAL_TEXTS:
                results_rows.append({"mode": mode, "avg_bits": avg_bits, "eval_dataset": ds_name,
                                     "perplexity": None, "accuracy": None,
                                     "quant_error_mean": None,
                                     "attn_recon_error_mean": None,
                                     "mlp_recon_error_mean": None})
            print(f"{mode} @ {avg_bits}: -- (flat-bit mode undefined at a fractional budget)")
            flush_checkpoint(results_rows, per_layer_store)
            continue
        rows, per_layer = out
        results_rows.extend(rows)
        per_layer_store[key] = per_layer
        flush_checkpoint(results_rows, per_layer_store)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  [normalized_grouped] avg_bits=3, min_bits=3: budget 2.336e+08, used 2.336e+08 (3.00 avg), sensitivity 6.8755e+09
  [normalized_grouped] real parameter-weighted average: 3.000 bits

=== pass: jab @ 3 avg bits ===
  layer  0/12: QKV 3 bits | MLP 3 bits
  layer  1/12: QKV 3 bits | MLP 3 bits
  layer  2/12: QKV 3 bits | MLP 3 bits
  layer  3/12: QKV 3 bits | MLP 3 bits
  layer  4/12: QKV 3 bits | MLP 3 bits
  layer  5/12: QKV 3 bits | MLP 3 bits
  layer  6/12: QKV 3 bits | MLP 3 bits
  layer  7/12: QKV 3 bits | MLP 3 bits
  layer  8/12: QKV 3 bits | MLP 3 bits
  layer  9/12: QKV 3 bits | MLP 3 bits
  layer 10/12: QKV 3 bits | MLP 3 bits
  layer 11/12: QKV 3 bits | MLP 3 bits
    [vram after 'jab @ 3 avg bits': 0.70 GB now, 1.48 GB peak this pass]
=== jab @ 3 avg bits: perplexity 141.821 ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  [normalized_grouped] avg_bits=3.5, min_bits=3: budget 2.725e+08, used 2.725e+08 (3.50 avg), sensitivity 1.7188e+09
  [normalized_grouped] real parameter-weighted average: 3.500 bits

=== pass: jab @ 3.5 avg bits ===
  layer  0/12: QKV Q4/K4/V3 bits | MLP 3 bits
  layer  1/12: QKV Q4/K4/V3 bits | MLP 3 bits
  layer  2/12: QKV 4 bits | MLP 3 bits
  layer  3/12: QKV Q8/K4/V3 bits | MLP 3 bits
  layer  4/12: QKV Q8/K4/V3 bits | MLP 3 bits
  layer  5/12: QKV Q4/K4/V3 bits | MLP 3 bits
  layer  6/12: QKV Q4/K4/V3 bits | MLP 3 bits
  layer  7/12: QKV Q4/K4/V3 bits | MLP 3 bits
  layer  8/12: QKV Q4/K4/V3 bits | MLP 4 bits
  layer  9/12: QKV Q4/K4/V3 bits | MLP 4 bits
  layer 10/12: QKV Q4/K4/V3 bits | MLP 4 bits
  layer 11/12: QKV 4 bits | MLP 4 bits
    [vram after 'jab @ 3.5 avg bits': 0.70 GB now, 1.48 GB peak this pass]
=== jab @ 3.5 avg bits: perplexity 90.712 ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  [normalized_grouped] avg_bits=4, min_bits=3: budget 3.114e+08, used 3.114e+08 (4.00 avg), sensitivity 8.9910e+08
  [normalized_grouped] real parameter-weighted average: 4.000 bits

=== pass: jab @ 4 avg bits ===
  layer  0/12: QKV Q4/K4/V3 bits | MLP fc4/proj3 bits
  layer  1/12: QKV Q4/K4/V3 bits | MLP fc4/proj3 bits
  layer  2/12: QKV Q8/K8/V3 bits | MLP fc4/proj3 bits
  layer  3/12: QKV Q8/K8/V3 bits | MLP fc4/proj3 bits
  layer  4/12: QKV Q8/K8/V3 bits | MLP fc4/proj3 bits
  layer  5/12: QKV Q8/K4/V3 bits | MLP 4 bits
  layer  6/12: QKV Q8/K4/V3 bits | MLP 4 bits
  layer  7/12: QKV Q4/K4/V3 bits | MLP 4 bits
  layer  8/12: QKV Q4/K4/V3 bits | MLP 4 bits
  layer  9/12: QKV Q4/K4/V3 bits | MLP 4 bits
  layer 10/12: QKV Q4/K4/V3 bits | MLP 4 bits
  layer 11/12: QKV Q4/K4/V3 bits | MLP 4 bits
    [vram after 'jab @ 4 avg bits': 0.70 GB now, 1.49 GB peak this pass]
=== jab @ 4 avg bits: perplexity 37.319 ===


... (40 outputs left out)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  [normalized_grouped] avg_bits=4.5, min_bits=3: budget 3.504e+08, used 3.504e+08 (4.50 avg), sensitivity 5.0456e+08
  [normalized_grouped] real parameter-weighted average: 4.500 bits

=== pass: adaptive_joint @ 4.5 avg bits ===
  layer  0/12: QKV Q4/K4/V3 bits, joint[QKV+MLP] FT 4.853706 -> 1.238403, flip rate 0.165 | MLP 4 bits, joint[QKV+MLP] FT 4.853706 -> 1.238403, flip rate 0.159
  layer  1/12: QKV Q4/K4/V3 bits, joint[QKV+MLP] FT 2.117915 -> 0.746673, flip rate 0.191 | MLP 4 bits, joint[QKV+MLP] FT 2.117915 -> 0.746673, flip rate 0.155
  layer  2/12: QKV Q8/K8/V4 bits, joint[QKV+MLP] FT 0.096498 -> 0.096498, flip rate 0.000 | MLP 4 bits, joint[QKV+MLP] FT 0.096498 -> 0.096498, flip rate 0.000
  layer  3/12: QKV Q8/K8/V3 bits, joint[QKV+MLP] FT 0.165660 -> 0.165660, flip rate 0.000 | MLP 4 bits, joint[QKV+MLP] FT 0.165660 -> 0.165660, flip rate 0.000
  layer  4/12: QKV Q8/K8/V3 bits, joint[QKV+MLP] FT 0.130301 -> 0.130301, flip rate 0.000 | MLP 4 bits, joint[QKV+MLP] FT 0.130301 

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  [normalized_grouped] avg_bits=6, min_bits=3: budget 4.671e+08, used 4.671e+08 (6.00 avg), sensitivity 1.2047e+08
  [normalized_grouped] real parameter-weighted average: 6.000 bits

=== pass: adaptive_joint @ 6 avg bits ===
  layer  0/12: QKV Q8/K8/V3 bits, joint[QKV+MLP] FT 0.086779 -> 0.034701, flip rate 0.275 | MLP 4 bits, joint[QKV+MLP] FT 0.086779 -> 0.034701, flip rate 0.121
  layer  1/12: QKV Q8/K8/V3 bits, joint[QKV+MLP] FT 0.038287 -> 0.038287, flip rate 0.000 | MLP 4 bits, joint[QKV+MLP] FT 0.038287 -> 0.038287, flip rate 0.000
  layer  2/12: QKV Q8/K8/V4 bits, joint[QKV+MLP] FT 0.107302 -> 0.107302, flip rate 0.000 | MLP 4 bits, joint[QKV+MLP] FT 0.107302 -> 0.107302, flip rate 0.000
  layer  3/12: QKV Q8/K8/V4 bits, joint[QKV+MLP] FT 0.215627 -> 0.215627, flip rate 0.000 | MLP 4 bits, joint[QKV+MLP] FT 0.215627 -> 0.215627, flip rate 0.000
  layer  4/12: QKV Q8/K8/V4 bits, joint[QKV+MLP] FT 0.123959 -> 0.123959, flip rate 0.000 | MLP 4 bits, joint[QKV+MLP] FT 0.123959 -> 0

## 13. Results: tidy table, `results.csv`, `results_per_layer.json`

In [23]:
FIELDNAMES = ["mode", "avg_bits", "eval_dataset", "perplexity", "accuracy",
              "quant_error_mean", "attn_recon_error_mean", "mlp_recon_error_mean"]


def fmt(v, spec):
    return "--" if v is None else format(v, spec)


print(f"{MODEL_ID}  ({ATTN.n_layers} layers, hidden {ATTN.hidden_size}, {ATTN.num_heads} heads, "
      f"intermediate {MLP.intermediate_size})")
print(f"\n{'mode':<16}{'avg_bits':>10}{'eval_dataset':>14}{'perplexity':>13}{'accuracy':>11}"
      f"{'quant_err':>12}{'attn_err':>12}{'mlp_err':>12}")
for r in results_rows:
    avg_bits_str = "--" if r["avg_bits"] is None else str(r["avg_bits"])
    print(f"{r['mode']:<16}{avg_bits_str:>10}{r['eval_dataset']:>14}"
          f"{fmt(r['perplexity'], '.3f'):>13}{fmt(r['accuracy'], '.4f'):>11}"
          f"{fmt(r['quant_error_mean'], '.4f'):>12}"
          f"{fmt(r['attn_recon_error_mean'], '.4f'):>12}"
          f"{fmt(r['mlp_recon_error_mean'], '.4f'):>12}")

with open(os.path.join(RUN_DIR, "results.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(results_rows)
print("\nSaved", os.path.join(RUN_DIR, "results.csv"))

with open(os.path.join(RUN_DIR, "results_per_layer.json"), "w") as f:
    json.dump(per_layer_store, f, indent=2)
print("Saved", os.path.join(RUN_DIR, "results_per_layer.json"),
      f"({len(per_layer_store)} arms, each with per-layer quant_error + attn_recon + mlp_recon "
      f"vectors)")


gpt2  (12 layers, hidden 768, 12 heads, intermediate 3072)

mode              avg_bits  eval_dataset   perplexity   accuracy   quant_err    attn_err     mlp_err
fp32                    --     wikitext2       24.357     0.4148      0.0000      0.0000      0.0000
fp32                    --            c4       32.770     0.3736      0.0000      0.0000      0.0000
jab                      3     wikitext2      141.821     0.2322      0.3606      0.4136      0.4639
jab                      3            c4      155.246     0.2197      0.3606      0.4001      0.4475
jab                    3.5     wikitext2       90.712     0.2784      0.2629      0.2980      0.3746
jab                    3.5            c4       90.315     0.2685      0.2629      0.2901      0.3647
jab                      4     wikitext2       37.319     0.3694      0.1999      0.3472      0.2388
jab                      4            c4       46.734     0.3375      0.1999      0.3437      0.2307
jab                    4.5     

## 14. Compare results: pivot table + compression view

In [ ]:
# --- pivot table: perplexity by mode x avg_bits, one table per eval dataset ---
for ds_name in EVAL_TEXTS:
    print(f"\n--- {ds_name} ---")
    header = f"{'avg_bits':>10}" + "".join(f"{m:>16}" for m in MODES)
    print(header)
    for avg_bits in SWEEP_BITS:
        row = f"{avg_bits:>10}"
        for mode in MODES:
            match = [r for r in results_rows
                     if r["mode"] == mode and r["avg_bits"] == avg_bits
                     and r["eval_dataset"] == ds_name]
            v = match[0]["perplexity"] if match and match[0]["perplexity"] is not None else None
            row += f"{v:>16.3f}" if v is not None else f"{'--':>16}"
        print(row)

print("\n--- compression ---")
REF_BITS = 16
# Non-embedding weight total: attn.c_attn + attn.c_proj + mlp.c_fc + mlp.c_proj per layer.
ATTN_C_ATTN_NUMEL = ATTN.qkv_numel                     # hidden * 3 * hidden
ATTN_C_PROJ_NUMEL = ATTN.hidden_size * ATTN.hidden_size
NONEMB_PARAMS = (ATTN_C_ATTN_NUMEL + ATTN_C_PROJ_NUMEL + MLP.mlp_numel) * ATTN.n_layers
# Every arm quantizes c_attn (QKV block) AND c_fc+c_proj (MLP block). c_proj of the ATTENTION
# sub-block (attn.c_proj) is never a target -- it's included in NONEMB_PARAMS only.
Q_PARAMS = (ATTN.qkv_numel + MLP.mlp_numel) * ATTN.n_layers

print(f"{MODEL_ID}: {NONEMB_PARAMS:,} non-embedding weight parameters "
      f"(attn.c_attn + attn.c_proj + mlp.c_fc + mlp.c_proj; biases, LayerNorms and "
      f"embeddings excluded -- none of them is a quantization target)")
print(f"  this notebook quantizes QKV + MLP: {Q_PARAMS:,} "
      f"({100 * Q_PARAMS / NONEMB_PARAMS:.1f}% of non-embedding weights)\n")


def bits_map_for(mode, avg_bits):
    """Reconstructs the per-unit bit assignment this arm actually used, without re-running it.
    Returns {layer_{i}_{QKV,MLP}: bits}.
    """
    if mode in ("uniform", "joint"):
        b = int(avg_bits)
        return {unit_name(i, m): b for i in range(ATTN.n_layers) for m in MATRICES}
    if mode == "type_offset":
        # Free, GPU-free, deterministic re-solve -- no scoring pass involved.
        return type_offset_assignment_for(avg_bits, min_bits=ALLOC_MIN_BITS)
    if mode == "jab_raw":
        return jab_raw_assignment_for(avg_bits, min_bits=ALLOC_MIN_BITS)
    if mode == "jab_prop":
        return jab_prop_assignment_for(avg_bits, min_bits=ALLOC_MIN_BITS)
    # jab / adaptive_joint: re-solve the greedy allocation (free, GPU-free) against the cached
    # section-11 scoring pass -- the SAME call run_arm makes (group-normalized, floored,
    # parameter-weighted). (This used to re-solve the raw `jab_assignment_for` instead, which is
    # not what the sweep ran.)
    return jab_assignment_for_normalized_grouped(avg_bits, min_bits=ALLOC_MIN_BITS)


print(f"{'mode':<16}{'avg_bits':>10}{'q-params':>14}{'actual avg':>12}{'compress':>10}")
seen = set()
for r in results_rows:
    key = (r["mode"], r["avg_bits"])
    if key in seen or r["perplexity"] is None:
        continue
    seen.add(key)
    bits_map = bits_map_for(*key)
    total_bits_quantized = sum(UNIT_PARAM_COUNTS[n] * b for n, b in bits_map.items())
    actual_avg_bits = total_bits_quantized / Q_PARAMS
    total_bits = total_bits_quantized + (NONEMB_PARAMS - Q_PARAMS) * REF_BITS
    ratio = (NONEMB_PARAMS * REF_BITS) / total_bits
    print(f"{r['mode']:<16}{r['avg_bits']:>10}{Q_PARAMS:>14,}"
          f"{actual_avg_bits:>12.2f}{ratio:>9.2f}x")

print("\n--- mean assigned bits per matrix type (average over layers) ---")
print(f"{'mode':<16}{'avg_bits':>10}" + "".join(f"{m:>8}" for m in MATRICES))
seen = set()
for r in results_rows:
    key = (r["mode"], r["avg_bits"])
    if key in seen or r["perplexity"] is None:
        continue
    seen.add(key)
    bits_map = bits_map_for(*key)
    means = [sum(bits_map[unit_name(i, m)] for i in range(ATTN.n_layers)) / ATTN.n_layers
             for m in MATRICES]
    print(f"{r['mode']:<16}{r['avg_bits']:>10}" + "".join(f"{v:>8.2f}" for v in means))


## 15. Test: grouped-normalized allocation + joint fine-tune

Allocates at `avg_bits=3.5, min_bits=3` on the **group-normalized** scores, checks the per-layer
assignment is non-uniform and never dips below the floor, then runs the full pipeline with
`finetune=True` and compares perplexity against uniform 3-bit and uniform 4-bit baselines (also
joint fine-tuned), which bracket the 3.5 budget.

Independent of the section-12 sweep -- it does not touch `results.json`.

In [ ]:
# --- 15a. the allocation itself: group-normalized scores + a 3-bit floor -------------------
TEST_AVG_BITS      = 3.5
TEST_MIN_BITS      = 3
TEST_UNIFORM_ARMS  = [3, 4]      # uniform baselines bracketing 3.5, both with the joint FT

assert ALLOC_MIN_BITS == TEST_MIN_BITS, (
    f"run_arm allocates with ALLOC_MIN_BITS={ALLOC_MIN_BITS}; set it to {TEST_MIN_BITS} so this "
    f"test and the sweep agree")

print(f"Allocating at avg_bits={TEST_AVG_BITS}, min_bits={TEST_MIN_BITS} on the "
      f"GROUP-NORMALIZED scores:")
test_assignment = jab_assignment_for_normalized_grouped(TEST_AVG_BITS, min_bits=TEST_MIN_BITS)

print(f"\n{'layer':>5}" + "".join(f"{m:>7}" for m in MATRICES))
for _i in range(ATTN.n_layers):
    print(f"{_i:>5}" + "".join(f"{test_assignment[unit_name(_i, m)]:>7}" for m in MATRICES))

_all_bits = list(test_assignment.values())
for _m in MATRICES:
    _mb = [test_assignment[unit_name(_i, _m)] for _i in range(ATTN.n_layers)]
    print(f"{_m:>5} widths: {dict(sorted(Counter(_mb).items()))}")
print(f"min width assigned: {min(_all_bits)} (floor was {TEST_MIN_BITS})")

assert min(_all_bits) >= TEST_MIN_BITS, (
    f"min_bits floor violated: got {min(_all_bits)} bits somewhere")
assert len(set(_all_bits)) > 1, (
    "allocation collapsed to a single uniform bit-width -- the group normalization is not "
    "producing per-layer variation; check that normalize_allocator_input_grouped divides by a "
    "SHARED per-block-type reference and not by each unit's own top-bit score")
print("\nOK: allocation respects the floor AND is non-uniform across layers.")


# --- 15b. end-to-end: adaptive+joint-FT vs. uniform+joint-FT --------------------------------
# Every arm below runs the FULL pipeline (GPTQ warm start -> joint block_loss fine-tune ->
# perplexity + accuracy on both eval sets), so this is minutes-to-hours, not seconds.
test_results = {}

print("\n" + "=" * 78)
print(f"ARM 1/{1 + len(TEST_UNIFORM_ARMS)}: adaptive_joint @ {TEST_AVG_BITS} avg bits "
      f"(min_bits={TEST_MIN_BITS}), joint fine-tune")
print("=" * 78)
_out = run_arm("adaptive_joint", TEST_AVG_BITS)
test_results[f"adaptive_joint @ {TEST_AVG_BITS} (min {TEST_MIN_BITS})"] = _out

for _k, _b in enumerate(TEST_UNIFORM_ARMS):
    print("\n" + "=" * 78)
    print(f"ARM {_k + 2}/{1 + len(TEST_UNIFORM_ARMS)}: uniform {_b}-bit baseline, "
          f"joint fine-tune")
    print("=" * 78)
    test_results[f"uniform {_b}-bit + joint FT"] = run_arm("joint", _b)


# --- 15c. side-by-side ----------------------------------------------------------------------
print("\n" + "=" * 78)
print("RESULT: perplexity (and top-1 accuracy) by arm, on both eval sets")
print("=" * 78)
print(f"{'arm':<40}{'wikitext2 ppl':>15}{'c4 ppl':>12}{'wt2 acc':>10}{'c4 acc':>10}")
print("-" * 87)
for _name, _out in test_results.items():
    if _out is None:
        print(f"{_name:<40}{'(undefined arm)':>15}")
        continue
    _rows, _ = _out
    _by_ds = {r["eval_dataset"]: r for r in _rows}
    print(f"{_name:<40}"
          f"{_by_ds['wikitext2']['perplexity']:>15.3f}"
          f"{_by_ds['c4']['perplexity']:>12.3f}"
          f"{_by_ds['wikitext2']['accuracy']:>10.4f}"
          f"{_by_ds['c4']['accuracy']:>10.4f}")

print("\nPer-layer commit decisions for the adaptive arm (one JOINT decision per layer -- QKV "
      "and MLP always agree now):")
_adaptive = test_results[f"adaptive_joint @ {TEST_AVG_BITS} (min {TEST_MIN_BITS})"]
if _adaptive is not None:
    _, _per_layer = _adaptive
    print(f"{'layer':>5}  {'QKV (Q,K,V)':>18}  {'MLP (fc,proj)':>18}")
    _fmt = lambda b: "/".join(map(str, b)) if isinstance(b, (tuple, list)) else str(b)
    for _i in range(ATTN.n_layers):
        _q = _per_layer["notes_qkv"].get(str(_i))
        _m = _per_layer["notes_mlp"].get(str(_i))
        _qs = f"{_fmt(_q[0])}b/{'FT' if _q[1] else 'GPTQ'}" if _q else "-"
        _ms = f"{_fmt(_m[0])}b/{'FT' if _m[1] else 'GPTQ'}" if _m else "-"
        print(f"{_i:>5}  {_qs:>18}  {_ms:>18}")
